# Appearance AI — Vision-Language Model Experiments

**Documented Google Colab notebook · Research workflow / experiment history**

This notebook records a controlled workflow for asking four vision-language models to describe images using **exactly five English descriptors**, validating response format, retaining raw output logs, and exporting structured results.

| Study | Image archive | Images | Model targets | Maximum response slots |
|---|---|---:|---|---:|
| Part 1 (earlier study) | `images.zip` | 75 | Gemini, Qwen, Llama, Mistral | 300 |
| Part 2 (main study) | `images2.zip` | 90 | Gemini, Qwen, Llama, Mistral | 360 |

The models configured in the supplied research code are **Gemini 3.5 Flash-Lite, Qwen3.8-27B, Llama 4 Scout 17B-16E Instruct, and Mistral Small 3.1 24B Instruct**. Identifiers are recorded in the experiment configuration cells and should be checked against the API logs before reporting precise executed versions.

> **Important:** This is an annotated record of the original workflow, **not** a one-click, top-to-bottom pipeline. Several later code cells are mutually alternative historical retries or manual corrections. Do **not** use **Run all** on a connected production dataset or active API credentials.

### Research question and output

- **Input:** an image and a versioned prompt.
- **Requested output:** five comma-separated English words describing the material or object's visible appearance.
- **Recording:** original response text, parsed five-word string, validity flag, model, image identity, timestamps, and error metadata where available.
- **Analysis:** use stored responses to examine the distribution and breadth of appearance descriptors. **Response-format validity and descriptor coverage are not visual accuracy measures.**

### Data flow

```text
images.zip (75) / images2.zip (90)
             |
       ZIP integrity check
             |
       image manifest + hashes
             |
       versioned prompt + model config
             |
   external Python batch runners / provider APIs
             |
      timestamped raw JSON response logs
             |
   format validation / independent audits
             |
       CSV / Excel research exports
```

### Operational prerequisites

1. Use a **Google Colab Python runtime** and a Drive project under `/content/drive_project_account/MyDrive/Translucency_AI/` (or adapt the project paths consistently).
2. Upload the appropriate archive to `Translucency_AI/input/` and keep its contents unchanged after creating the manifest.
3. Configure the necessary **Colab Secrets**. Relevant names include `GEMINI_API`, `Flare_api_key`, `Flare_account_id`, and optional alternative Cloudflare account pairs (`Flare2_*`, `Flare3_*`). **Never store real secret values in the notebook or repository.**
4. Supply the **external `.py` runners** used by some cells; they are not embedded in this `.ipynb` file. See the dependency notes at the end.
5. Examine saved logs and existing audit results before initiating any API retry to avoid unwanted extra usage.

### How to read the notebook

- **Offline:** validates archives, computes manifests, reads logs, or exports saved results without model calls.
- **LIVE API:** may send paid or quota-limited requests; run intentionally.
- **Historical/optional:** preserves the actual troubleshooting or recovery steps. These are **alternatives**, not consecutive required steps.
- **Manual correction:** can change local output artifacts and requires independent review.

**Navigation:** §1 Environment · §2 Part 1 / 75 images · §3 Part 2 / 90 images · §4 Provider retries · §5 Format repair · §6 Export.


## 1. Colab environment and project storage

The setup cells below install dependencies, mount Google Drive, and establish the expected dataset location.

### 1.1 — Install dependencies
**Type:** Setup · **External requests:** package downloads only

Installs libraries used for authentication, dataset inspection, API access, tables, Excel files, and image validation. Execute once at the start of a fresh Google Colab session. The libraries are part of the runtime, not files to commit to GitHub.

In [ ]:
# Install dependencies in the current Colab runtime before running project cells.
# Colab environments are temporary; packages may require reinstalling after a reset.
%pip install -q \
    "google-auth==2.49.0" \
    "pandas==2.2.3" \
    "google-genai" \
    "openai" \
    "openpyxl" \
    "pillow" \
    "tqdm" \
    "natsort"


### 1.2 — Mount Google Drive and locate the image archive
**Type:** Offline setup · **API inference:** No

Mount the research Google Drive at `/content/drive_project_account`, search the expected project locations, and detect `images.zip`. The next stage uses fixed project paths; confirm that the printed directory matches your Drive layout before proceeding.

In [ ]:

# ==========================================================
# APPEARANCE AI - STEP 2
# GOOGLE DRIVE CONNECTION & DATASET CHECK
# ==========================================================

from google.colab import drive
from pathlib import Path

print("=" * 65)
print("APPEARANCE AI - GOOGLE DRIVE SETUP")
print("=" * 65)

# Mount the correct Google Drive account
# This project uses a custom Drive mount path rather than /content/drive.
# Keep it consistent with the absolute paths used by subsequent cells.
MOUNT_POINT = "/content/drive_project_account"

drive.mount(MOUNT_POINT)

MYDRIVE = Path(MOUNT_POINT) / "MyDrive"

# Check possible project locations
PROJECT_CANDIDATES = [
    MYDRIVE / "Appearance_AI",
    MYDRIVE / "Translucency_AI",
]

print("\nCHECKING PROJECT DIRECTORIES")
print("-" * 65)

found_datasets = []

for project in PROJECT_CANDIDATES:

    zip_file = project / "input" / "images.zip"

    print("\nProject:", project.name)
    print("Folder exists:", project.is_dir())
    print("images.zip exists:", zip_file.is_file())

    if zip_file.is_file():
        print("ZIP size (MB):",
              round(zip_file.stat().st_size / 1024**2, 2))

        found_datasets.append((project, zip_file))

print("\n" + "=" * 65)
print("DATASET DETECTION RESULTS")
print("=" * 65)

if len(found_datasets) == 1:

    PROJECT_ROOT, ZIP_PATH = found_datasets[0]

    INPUT_DIR = PROJECT_ROOT / "input"
    LOGS_DIR = PROJECT_ROOT / "logs"
    RESULTS_DIR = PROJECT_ROOT / "results"

    print("STATUS: DATASET FOUND")
    print("Project:", PROJECT_ROOT)
    print("ZIP:", ZIP_PATH)

elif len(found_datasets) == 0:

    print("STATUS: DATASET NOT FOUND")
    print("Please check the Google Drive input folder.")

else:

    print("STATUS: MULTIPLE DATASETS FOUND")
    print("Choose the correct project before proceeding.")

    for project, zip_file in found_datasets:
        print("-", zip_file)

print("=" * 65)


---
## 2. Part 1 — Initial 75-image experiment

This part records an earlier V2 experiment. It has its **own dataset, manifest, API logs, recovery scripts, and Excel output**. It should not be confused with or automatically merged into the 90-image study.

### 2.1 — Validate the 75-image dataset and create a manifest
**Type:** Offline validation · **Expected input:** `Translucency_AI/input/images.zip`

Validate that the archive is readable, contains 75 `.jfif` images numbered `001`–`075`, and contains no duplicate IDs. Read each image to record dimensions, format, byte count, and MD5 checksum. Write `manifest_images75.csv` without silently overwriting a different manifest.

**Why it matters:** the manifest ties each model response to a specific image file and makes accidental dataset changes visible.

In [ ]:

# ============================================================
# APPEARANCE AI V2 - STEP 3
# DATASET 75 VALIDATION + MANIFEST
# ============================================================

from pathlib import Path, PurePosixPath
from zipfile import ZipFile
from collections import Counter
from PIL import Image
import pandas as pd
import hashlib
import io

# ------------------------------------------------------------
# 1. PATH CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive_project_account/MyDrive/Translucency_AI"
)

ZIP_PATH = PROJECT_ROOT / "input" / "images.zip"

# Separate workspace for the new experiment
V2_ROOT = PROJECT_ROOT / "Appearance_AI_V2"

LOGS_DIR = V2_ROOT / "logs"
RESULTS_DIR = V2_ROOT / "results"

MANIFEST_PATH = LOGS_DIR / "manifest_images75.csv"

# Part 1 is a strict 75-image dataset with numeric .jfif filenames.
# The generated manifest records identity, dimensions, and a checksum per image.
EXPECTED_COUNT = 75

# ------------------------------------------------------------
# 2. READ ZIP
# ------------------------------------------------------------

if not ZIP_PATH.is_file():
    raise FileNotFoundError(f"ZIP not found: {ZIP_PATH}")

rows = []

with ZipFile(ZIP_PATH, "r") as zf:

    bad_member = zf.testzip()

    if bad_member is not None:
        raise ValueError(
            f"Corrupted ZIP member: {bad_member}"
        )

    members = [
        item
        for item in zf.infolist()
        if not item.is_dir()
        and PurePosixPath(item.filename).suffix.lower()
        == ".jfif"
    ]

    print("=" * 65)
    print("APPEARANCE AI V2 - DATASET VALIDATION")
    print("=" * 65)
    print("ZIP:", ZIP_PATH.name)
    print("Images detected:", len(members))
    print("Expected:", EXPECTED_COUNT)

    if len(members) != EXPECTED_COUNT:
        raise ValueError(
            f"Expected {EXPECTED_COUNT} images, "
            f"found {len(members)}"
        )

    # Check exact filename sequence
    actual_names = [
        PurePosixPath(item.filename).name.lower()
        for item in members
    ]

    expected_names = [
        f"{i:03d}.jfif"
        for i in range(1, EXPECTED_COUNT + 1)
    ]

    counts = Counter(actual_names)

    duplicates = [
        name for name, count in counts.items()
        if count > 1
    ]

    missing = sorted(set(expected_names) - set(actual_names))
    unexpected = sorted(set(actual_names) - set(expected_names))

    if duplicates or missing or unexpected:
        print("Duplicates:", duplicates)
        print("Missing:", missing)
        print("Unexpected:", unexpected)
        raise ValueError("Image filename validation failed.")

    # --------------------------------------------------------
    # 3. IMAGE VERIFICATION AND MD5
    # --------------------------------------------------------

    for item in members:

        image_name = PurePosixPath(item.filename).name
        row_id = int(Path(image_name).stem)

        image_bytes = zf.read(item.filename)

        # Verify that PIL can decode the image
        with Image.open(io.BytesIO(image_bytes)) as img:
            image_format = img.format
            width, height = img.size
            img.verify()

        md5 = hashlib.md5(image_bytes).hexdigest()

        rows.append({
            "row_id": row_id,
            "image_name": image_name,
            "relative_path": item.filename,
            "md5": md5,
            "size_bytes": len(image_bytes),
            "width": width,
            "height": height,
            "image_format": image_format
        })

# ------------------------------------------------------------
# 4. CREATE MANIFEST
# ------------------------------------------------------------

manifest_df = pd.DataFrame(rows)

manifest_df = (
    manifest_df
    .sort_values("row_id")
    .reset_index(drop=True)
)

if manifest_df["row_id"].tolist() != list(range(1, 76)):
    raise ValueError("Image numbering is incorrect.")

# ------------------------------------------------------------
# 5. SAVE SAFELY
# ------------------------------------------------------------

LOGS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if MANIFEST_PATH.exists():

    existing = pd.read_csv(
        MANIFEST_PATH,
        dtype={"image_name": str, "md5": str}
    )

    if not existing.equals(manifest_df):
        raise FileExistsError(
            "An existing manifest differs from the "
            "current dataset. Nothing was overwritten."
        )

    print("\nExisting manifest verified. No overwrite.")

else:

    manifest_df.to_csv(
        MANIFEST_PATH,
        index=False,
        encoding="utf-8-sig"
    )

    print("\nNew manifest saved successfully.")

# ------------------------------------------------------------
# 6. REPORT
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("DATASET VALIDATION SUMMARY")
print("=" * 65)

print("Total images :", len(manifest_df))
print("First image  :", manifest_df.iloc[0]["image_name"])
print("Last image   :", manifest_df.iloc[-1]["image_name"])

print("Unique MD5   :", manifest_df["md5"].nunique())
print("Manifest     :", MANIFEST_PATH)

print("\nSTATUS: DATASET READY")
print("=" * 65)

display(manifest_df.head(10))


### 2.2 — Register Prompt V2 and the four model identifiers
**Type:** Offline configuration · **API inference:** No

Save the exact five-descriptor English prompt, its SHA-256 hash, dataset manifest hash, and model identifiers in `experiment_config_images75.json`. Verify availability of the Colab secrets without showing credential values.

**Output constraint:** five comma-separated English descriptive words, with no explanation or full sentence. A valid response format does **not** imply that the words accurately match the image.

In [ ]:

# ============================================================
# APPEARANCE AI V2 - STEP 4
# PROMPT REGISTRATION + MODEL CONFIG + API SECRET CHECK
# ============================================================

from pathlib import Path
from google.colab import userdata
import hashlib
import json
import pandas as pd

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive_project_account/MyDrive/Translucency_AI"
)

V2_ROOT = PROJECT_ROOT / "Appearance_AI_V2"

LOGS_DIR = V2_ROOT / "logs"
RESULTS_DIR = V2_ROOT / "results"

MANIFEST_PATH = LOGS_DIR / "manifest_images75.csv"

CONFIG_PATH = LOGS_DIR / "experiment_config_images75.json"

LOGS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. OFFICIAL PROMPT V2
# ------------------------------------------------------------

# Freeze the exact research prompt and hash so results can be checked
# against the intended instruction version during later processing.
PROMPT_VERSION = "V2"

PROMPT_V2 = (
    "Describe the visual appearance of the main object or "
    "material in this image using exactly five English "
    "descriptive words.\n\n"
    "Rules:\n"
    "- Output exactly five words\n"
    "- Separate them with commas\n"
    "- No explanation\n"
    "- No full sentence"
)

PROMPT_HASH = hashlib.sha256(
    PROMPT_V2.encode("utf-8")
).hexdigest()

# ------------------------------------------------------------
# 3. MODEL CONFIGURATION
# ------------------------------------------------------------

MODELS = {
    "gemini": "gemini-3.5-flash-lite",
    "qwen": "@cf/qwen/qwen3.8-27b",
    "llama": "@cf/meta/llama-4-scout-17b-16e-instruct",
    "mistral": "@cf/mistralai/mistral-small-3.1-24b-instruct"
}

# Model availability will be tested separately.

# ------------------------------------------------------------
# 4. VERIFY DATASET MANIFEST
# ------------------------------------------------------------

if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"Manifest not found: {MANIFEST_PATH}"
    )

manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={"image_name": str, "md5": str}
)

if len(manifest) != 75:
    raise ValueError(
        f"Expected 75 images, found {len(manifest)}"
    )

MANIFEST_HASH = hashlib.sha256(
    MANIFEST_PATH.read_bytes()
).hexdigest()

# ------------------------------------------------------------
# 5. REGISTER EXPERIMENT CONFIG
# ------------------------------------------------------------

experiment_config = {
    "experiment": "Appearance_AI_V2",
    "dataset": "images75",
    "image_count": 75,
    "zip_file": "images.zip",
    "prompt_version": PROMPT_VERSION,
    "prompt": PROMPT_V2,
    "prompt_sha256": PROMPT_HASH,
    "manifest_sha256": MANIFEST_HASH,
    "models": MODELS
}

# Prevent accidental replacement of an existing setup
if CONFIG_PATH.exists():

    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        previous_config = json.load(f)

    if previous_config != experiment_config:
        raise RuntimeError(
            "Existing experiment configuration differs. "
            "Nothing was overwritten."
        )

    print("Existing configuration verified.")

else:

    with open(CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(
            experiment_config,
            f,
            indent=2,
            ensure_ascii=False
        )

    print("New experiment configuration saved.")

# ------------------------------------------------------------
# 6. API SECRET CHECK
# ------------------------------------------------------------

SECRET_NAMES = {
    "Gemini API": "GEMINI_API",
    "Cloudflare API": "Flare_api_key",
    "Cloudflare Account": "Flare_account_id"
}

secret_status = {}

for label, secret_name in SECRET_NAMES.items():

    try:
        value = userdata.get(secret_name)
        secret_status[label] = bool(
            value and str(value).strip()
        )

    except Exception:
        secret_status[label] = False

# ------------------------------------------------------------
# 7. SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("APPEARANCE AI V2 - CONFIGURATION")
print("=" * 65)

print("\nDATASET")
print("Images:", len(manifest))
print("Manifest:", MANIFEST_PATH.name)

print("\nPROMPT")
print("Version:", PROMPT_VERSION)
print("-" * 65)
print(PROMPT_V2)

print("\n" + "-" * 65)
print("REGISTERED MODELS")

for label, model_id in MODELS.items():
    print(f"{label.upper():<10}: {model_id}")

print("\n" + "-" * 65)
print("API SECRET CHECK")

for label, ready in secret_status.items():
    print(f"{label:<22}: {ready}")

print("\n" + "-" * 65)
print("Configuration saved:", CONFIG_PATH)

all_ready = all(secret_status.values())

print("\nALL REQUIRED SECRETS READY:", all_ready)

print("=" * 65)


### 2.3 — Gemini single-image smoke test
**Type:** **LIVE Gemini API** if no cached success · **Test image:** `001.jfif`

Verify image checksum and prompt hash, then request a Gemini response for a single image. A format validator checks whether exactly five comma-separated descriptors match the expected character pattern; the raw response is preserved alongside the normalized response and validity flag.

The cell searches for a previous matching success before a new request. Re-running may still use quota if no valid cached record is found.

In [ ]:

# ============================================================
# APPEARANCE AI V2 - STEP 5
# GEMINI SINGLE IMAGE TEST
# Dataset: images75
# Image: 001.jfif
# ============================================================

import base64
import hashlib
import json
import re
import requests
import pandas as pd

from pathlib import Path
from zipfile import ZipFile
from datetime import datetime, timezone
from google.colab import userdata

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

ROOT = Path(
    "/content/drive_project_account/MyDrive/Translucency_AI"
)

V2_ROOT = ROOT / "Appearance_AI_V2"

ZIP_PATH = ROOT / "input/images.zip"

MANIFEST_PATH = (
    V2_ROOT / "logs/manifest_images75.csv"
)

CONFIG_PATH = (
    V2_ROOT / "logs/experiment_config_images75.json"
)

TEST_LOG_DIR = V2_ROOT / "logs/single_tests"
TEST_LOG_DIR.mkdir(parents=True, exist_ok=True)

ROW_ID = 1

# ------------------------------------------------------------
# 2. LOAD SAVED CONFIGURATION
# ------------------------------------------------------------

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

PROMPT = config["prompt"]
PROMPT_HASH = config["prompt_sha256"]
MODEL = config["models"]["gemini"]

actual_prompt_hash = hashlib.sha256(
    PROMPT.encode("utf-8")
).hexdigest()

if actual_prompt_hash != PROMPT_HASH:
    raise ValueError("Prompt hash mismatch!")

# ------------------------------------------------------------
# 3. LOAD IMAGE AND VERIFY MD5
# ------------------------------------------------------------

manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={"image_name": str, "md5": str}
)

matches = manifest.loc[
    manifest["row_id"].astype(int) == ROW_ID
]

if len(matches) != 1:
    raise ValueError("Image row missing or duplicated.")

row = matches.iloc[0]

IMAGE_NAME = str(row["image_name"])
ZIP_MEMBER = str(row["relative_path"])
EXPECTED_MD5 = str(row["md5"])

with ZipFile(ZIP_PATH, "r") as zf:
    image_bytes = zf.read(ZIP_MEMBER)

ACTUAL_MD5 = hashlib.md5(image_bytes).hexdigest()

if ACTUAL_MD5.lower() != EXPECTED_MD5.lower():
    raise ValueError("Image MD5 verification failed!")

# ------------------------------------------------------------
# 4. FIVE-WORD VALIDATOR
# ------------------------------------------------------------

def validate_five_words(raw_text):

    if not isinstance(raw_text, str):
        return "", False

    parts = [
        part.strip()
        for part in raw_text.strip().split(",")
    ]

    # Five comma-separated English descriptors.
    # Hyphenated adjectives count as one descriptor.

    word_pattern = re.compile(
        r"[A-Za-z]+(?:-[A-Za-z]+)*"
    )

    valid = (
        len(parts) == 5
        and all(
            word_pattern.fullmatch(part)
            for part in parts
        )
    )

    if valid:
        cleaned = ", ".join(
            part.lower() for part in parts
        )
    else:
        cleaned = ""

    return cleaned, valid

# ------------------------------------------------------------
# 5. CHECK FOR PREVIOUS SUCCESS
# ------------------------------------------------------------

# Look for a matching successful log before consuming more API quota.
# A match requires the model, prompt hash, and image checksum to agree.
previous_success = None

for old_path in sorted(
    TEST_LOG_DIR.glob("gemini_001_*.json")
):

    try:
        with open(old_path, "r", encoding="utf-8") as f:
            old = json.load(f)

        if (
            old.get("valid") is True
            and old.get("model") == MODEL
            and old.get("prompt_sha256") == PROMPT_HASH
            and old.get("image_md5") == ACTUAL_MD5
        ):
            previous_success = old
            break

    except (OSError, ValueError, TypeError):
        continue

# ------------------------------------------------------------
# 6. SEND API REQUEST ONLY IF NEEDED
# ------------------------------------------------------------

if previous_success is not None:

    print("Previous successful test found.")
    print("No new API request sent.")
    print("Raw:", previous_success["raw_response"])
    print("Valid:", previous_success["valid"])

else:

    GEMINI_API_KEY = userdata.get("GEMINI_API")

    if not GEMINI_API_KEY:
        raise ValueError("Gemini API key missing!")

    encoded_image = base64.b64encode(
        image_bytes
    ).decode("ascii")

    API_URL = (
        "https://generativelanguage.googleapis.com/"
        f"v1beta/models/{MODEL}:generateContent"
    )

    headers = {
        "x-goog-api-key": GEMINI_API_KEY,
        "Content-Type": "application/json"
    }

    payload = {
        "contents": [
            {
                "role": "user",
                "parts": [
                    {"text": PROMPT},
                    {
                        "inline_data": {
                            "mime_type": "image/jpeg",
                            "data": encoded_image
                        }
                    }
                ]
            }
        ],
        "generationConfig": {
            "temperature": 0,
            "maxOutputTokens": 512
        }
    }

    print("=" * 65)
    print("APPEARANCE AI V2 - GEMINI TEST")
    print("=" * 65)

    print("Image:", IMAGE_NAME)
    print("Model:", MODEL)
    print("MD5 verified:", True)
    print("\nSending request...")

    result = {
        "timestamp": datetime.now(
            timezone.utc
        ).isoformat(),
        "dataset": "images75",
        "row_id": ROW_ID,
        "image_name": IMAGE_NAME,
        "image_md5": ACTUAL_MD5,
        "model": MODEL,
        "prompt_version": "V2",
        "prompt_sha256": PROMPT_HASH,
        "raw_response": "",
        "five_words": "",
        "valid": False,
        "status": "pending",
        "generation_config": payload[
            "generationConfig"
        ]
    }

    try:

        response = requests.post(
            API_URL,
            headers=headers,
            json=payload,
            timeout=120
        )

        result["http_status"] = response.status_code

        try:
            api_data = response.json()
        except ValueError:
            api_data = {
                "non_json_response": response.text[:2000]
            }

        result["full_api_response"] = api_data

        if not response.ok:

            result["status"] = "api_error"
            result["error"] = str(
                api_data.get("error", api_data)
            )[:1500]

        else:

            candidates = api_data.get("candidates", [])

            text_parts = []

            if candidates:

                parts = (
                    candidates[0]
                    .get("content", {})
                    .get("parts", [])
                )

                text_parts = [
                    part["text"]
                    for part in parts
                    if isinstance(part, dict)
                    and isinstance(
                        part.get("text"), str
                    )
                ]

            raw = "\n".join(text_parts).strip()

            cleaned, valid = validate_five_words(raw)

            result["raw_response"] = raw
            result["five_words"] = cleaned
            result["valid"] = bool(valid)

            result["status"] = (
                "success" if valid
                else "invalid_format"
            )

            result["usage"] = api_data.get(
                "usageMetadata", {}
            )

    except requests.RequestException as exc:

        result["status"] = "request_error"
        result["error"] = (
            f"{type(exc).__name__}: {str(exc)}"
        )[:1500]

    # --------------------------------------------------------
    # 7. SAVE LOG WITHOUT OVERWRITING
    # --------------------------------------------------------

    stamp = datetime.now(
        timezone.utc
    ).strftime("%Y%m%d_%H%M%S_%f")

    LOG_PATH = (
        TEST_LOG_DIR /
        f"gemini_001_{stamp}.json"
    )

    with open(LOG_PATH, "x", encoding="utf-8") as f:

        json.dump(
            result,
            f,
            indent=2,
            ensure_ascii=False
        )

    # --------------------------------------------------------
    # 8. PRINT REPORT
    # --------------------------------------------------------

    print("\n" + "=" * 65)
    print("GEMINI TEST RESULT")
    print("=" * 65)

    print("Image       :", IMAGE_NAME)
    print("Model       :", MODEL)
    print("HTTP status :", result.get("http_status"))
    print("Status      :", result["status"])

    print("\nRaw response:")
    print(result["raw_response"])

    print("\nFive words:")
    print(result["five_words"])

    print("\nValid:", result["valid"])

    if "error" in result:
        print("\nError:", result["error"])

    print("\nLog saved:", LOG_PATH)
    print("=" * 65)


### 2.4 — Four-model smoke test on the same image
**Type:** **LIVE API** unless cached · **Models:** Gemini, Qwen, Llama, Mistral

Send the same image/prompt to the four configured models, using provider-specific request formats. Save independent timestamped JSON records, attempt reuse of valid prior results, stop on certain Cloudflare quota responses, and export one CSV comparison.

The `temperature`, token-limit, and reasoning options are passed in provider-specific payloads; do **not** assume that all APIs interpret or honor them identically.

In [ ]:

# ============================================================
# APPEARANCE AI V2 - STEP 6
# ALL FOUR MODELS - SINGLE IMAGE TEST
#
# Dataset: images75
# Image: 001.jfif
# Models: Gemini, Qwen, Llama, Mistral
#
# - Reuse successful previous tests
# - Preserve original model responses
# - Validate exactly five descriptors
# - Save separate JSON logs
# - Export one comparison CSV
# ============================================================

import base64
import hashlib
import json
import re
import time
from pathlib import Path
from zipfile import ZipFile
from datetime import datetime, timezone

import pandas as pd
import requests
from google.colab import userdata
from IPython.display import display


# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

ROOT = Path(
    "/content/drive_project_account/MyDrive/Translucency_AI"
)

V2_ROOT = ROOT / "Appearance_AI_V2"

ZIP_PATH = ROOT / "input/images.zip"

MANIFEST_PATH = (
    V2_ROOT / "logs/manifest_images75.csv"
)

CONFIG_PATH = (
    V2_ROOT / "logs/experiment_config_images75.json"
)

TEST_LOG_DIR = V2_ROOT / "logs/single_tests"

RESULTS_DIR = V2_ROOT / "results"

TEST_LOG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ROW_ID = 1

# Set True only when you intentionally want
# to repeat previously successful API tests.
FORCE_RETEST = False


# ------------------------------------------------------------
# 2. LOAD CONFIGURATION
# ------------------------------------------------------------

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

PROMPT = CONFIG["prompt"]
PROMPT_HASH = CONFIG["prompt_sha256"]
MODELS = CONFIG["models"]

assert hashlib.sha256(
    PROMPT.encode("utf-8")
).hexdigest() == PROMPT_HASH, "Prompt hash mismatch."

EXPECTED_MODELS = [
    "gemini",
    "qwen",
    "llama",
    "mistral"
]

for model_key in EXPECTED_MODELS:
    if model_key not in MODELS:
        raise ValueError(
            f"Missing model in config: {model_key}"
        )


# ------------------------------------------------------------
# 3. LOAD IMAGE FROM ZIP
# ------------------------------------------------------------

manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={"image_name": str, "md5": str}
)

target = manifest.loc[
    manifest["row_id"].astype(int) == ROW_ID
]

if len(target) != 1:
    raise ValueError("Image row missing or duplicated.")

target = target.iloc[0]

IMAGE_NAME = str(target["image_name"])
ZIP_MEMBER = str(target["relative_path"])
EXPECTED_MD5 = str(target["md5"])

with ZipFile(ZIP_PATH, "r") as zf:
    IMAGE_BYTES = zf.read(ZIP_MEMBER)

IMAGE_MD5 = hashlib.md5(
    IMAGE_BYTES
).hexdigest()

if IMAGE_MD5.lower() != EXPECTED_MD5.lower():
    raise ValueError("Image MD5 mismatch.")

IMAGE_DATA_URL = (
    "data:image/jpeg;base64,"
    + base64.b64encode(IMAGE_BYTES).decode("ascii")
)


# ------------------------------------------------------------
# 4. VALIDATION
# ------------------------------------------------------------

def validate_five_words(raw_text):

    if not isinstance(raw_text, str):
        return "", False

    parts = [
        item.strip()
        for item in raw_text.strip().split(",")
    ]

    pattern = re.compile(
        r"[A-Za-z]+(?:-[A-Za-z]+)*"
    )

    valid = (
        len(parts) == 5
        and all(
            pattern.fullmatch(item) is not None
            for item in parts
        )
    )

    cleaned = (
        ", ".join(item.lower() for item in parts)
        if valid else ""
    )

    return cleaned, bool(valid)


def timestamp_utc():

    return datetime.now(
        timezone.utc
    ).strftime("%Y%m%d_%H%M%S_%f")


# ------------------------------------------------------------
# 5. FIND PREVIOUS SUCCESSFUL TEST
# ------------------------------------------------------------

def find_previous_success(model_key, model_id):

    pattern = f"{model_key}_{ROW_ID:03d}_*.json"

    log_files = sorted(
        TEST_LOG_DIR.glob(pattern),
        reverse=True
    )

    for path in log_files:

        try:
            with open(path, "r", encoding="utf-8") as f:
                old = json.load(f)

            if (
                old.get("model") == model_id
                and old.get("image_md5") == IMAGE_MD5
                and old.get("prompt_sha256") == PROMPT_HASH
                and old.get("valid") is True
                and old.get("status") == "success"
            ):
                return old

        except (OSError, ValueError, TypeError):
            continue

    return None


# ------------------------------------------------------------
# 6. LOAD SECRETS
# ------------------------------------------------------------

def get_required_secret(secret_name):

    try:
        value = userdata.get(secret_name)
    except Exception as exc:
        raise RuntimeError(
            f"Cannot read secret '{secret_name}': "
            f"{type(exc).__name__}"
        ) from exc

    if not value or not str(value).strip():
        raise ValueError(
            f"Secret '{secret_name}' is missing."
        )

    return str(value).strip()


GEMINI_KEY = get_required_secret("GEMINI_API")

CF_KEY = get_required_secret("Flare_api_key")

CF_ACCOUNT = get_required_secret("Flare_account_id")


# ------------------------------------------------------------
# 7. GEMINI REST REQUEST
# ------------------------------------------------------------

def request_gemini(model_id):

    url = (
        "https://generativelanguage.googleapis.com/"
        f"v1beta/models/{model_id}:generateContent"
    )

    headers = {
        "x-goog-api-key": GEMINI_KEY,
        "Content-Type": "application/json"
    }

    payload = {
        "contents": [
            {
                "role": "user",
                "parts": [
                    {"text": PROMPT},
                    {
                        "inline_data": {
                            "mime_type": "image/jpeg",
                            "data": base64.b64encode(
                                IMAGE_BYTES
                            ).decode("ascii")
                        }
                    }
                ]
            }
        ],
        "generationConfig": {
            "temperature": 0,
            "maxOutputTokens": 512
        }
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload,
        timeout=120
    )

    try:
        data = response.json()
    except ValueError:
        data = {"non_json": response.text[:2000]}

    raw = ""

    if response.ok:

        candidates = data.get("candidates", [])

        if candidates:

            parts = (
                candidates[0]
                .get("content", {})
                .get("parts", [])
            )

            raw = "\n".join(
                p["text"]
                for p in parts
                if isinstance(p, dict)
                and isinstance(p.get("text"), str)
            ).strip()

    return {
        "http_status": response.status_code,
        "api_data": data,
        "raw_response": raw,
        "usage": data.get("usageMetadata", {})
    }


# ------------------------------------------------------------
# 8. CLOUDFLARE REST REQUEST
# ------------------------------------------------------------

def request_cloudflare(model_key, model_id):

    url = (
        "https://api.cloudflare.com/client/v4/accounts/"
        f"{CF_ACCOUNT}/ai/run/{model_id}"
    )

    headers = {
        "Authorization": f"Bearer {CF_KEY}",
        "Content-Type": "application/json"
    }

    content = [
        {
            "type": "text",
            "text": PROMPT
        },
        {
            "type": "image_url",
            "image_url": {
                "url": IMAGE_DATA_URL
            }
        }
    ]

    payload = {
        "messages": [
            {
                "role": "user",
                "content": content
            }
        ],
        "stream": False
    }

    # Model-specific generation settings
    if model_key == "qwen":

        payload["reasoning_effort"] = "low"
        payload["max_completion_tokens"] = 256
        payload["temperature"] = 0

    else:

        payload["max_tokens"] = 128
        payload["temperature"] = 0

    response = requests.post(
        url,
        headers=headers,
        json=payload,
        timeout=180
    )

    try:
        data = response.json()
    except ValueError:
        data = {"non_json": response.text[:2000]}

    raw = ""
    usage = {}

    result = data.get("result", {})

    if response.ok and data.get("success") is not False:

        if isinstance(result, str):
            raw = result

        elif isinstance(result, dict):

            raw = result.get("response", "") or ""

            if not raw:
                choices = result.get("choices", [])

                if choices:
                    raw = (
                        choices[0]
                        .get("message", {})
                        .get("content", "")
                    ) or ""

            usage = result.get("usage", {})

        # Some endpoints return OpenAI-style JSON
        if not raw and data.get("choices"):

            raw = (
                data["choices"][0]
                .get("message", {})
                .get("content", "")
            ) or ""

        if not usage:
            usage = data.get("usage", {})

    if not isinstance(raw, str):
        raw = ""

    return {
        "http_status": response.status_code,
        "api_data": data,
        "raw_response": raw,
        "usage": usage
    }


# ------------------------------------------------------------
# 9. TEST ONE MODEL
# ------------------------------------------------------------

def test_model(model_key, model_id):

    print("\n" + "=" * 65)
    print(f"TESTING {model_key.upper()}")
    print("=" * 65)

    print("Image:", IMAGE_NAME)
    print("Model:", model_id)

    if not FORCE_RETEST:

        previous = find_previous_success(
            model_key, model_id
        )

        if previous is not None:

            print("Status: PREVIOUS SUCCESS REUSED")
            print("Raw:", previous.get("raw_response"))
            print("Valid:", previous.get("valid"))
            print("No API request sent.")

            return previous

    record = {
        "timestamp": datetime.now(
            timezone.utc
        ).isoformat(),
        "dataset": "images75",
        "row_id": ROW_ID,
        "image_name": IMAGE_NAME,
        "image_md5": IMAGE_MD5,
        "model_key": model_key,
        "model": model_id,
        "prompt_version": "V2",
        "prompt_sha256": PROMPT_HASH,
        "status": "pending",
        "http_status": None,
        "raw_response": "",
        "five_words": "",
        "valid": False,
        "error": "",
        "usage": {}
    }

    try:

        print("Sending API request...")

        if model_key == "gemini":

            output = request_gemini(model_id)

        else:

            output = request_cloudflare(
                model_key, model_id
            )

        record["http_status"] = output["http_status"]
        record["api_response"] = output["api_data"]
        record["usage"] = output["usage"]

        api_data = output["api_data"]

        if (
            output["http_status"] != 200
            or api_data.get("success") is False
        ):

            record["status"] = "api_error"

            error_obj = api_data.get(
                "errors",
                api_data.get("error", api_data)
            )

            record["error"] = str(error_obj)[:1500]

        else:

            raw = output["raw_response"]

            cleaned, valid = validate_five_words(
                raw
            )

            record["raw_response"] = raw
            record["five_words"] = cleaned
            record["valid"] = valid

            record["status"] = (
                "success" if valid
                else "invalid_format"
            )

    except requests.RequestException as exc:

        record["status"] = "request_error"
        record["error"] = (
            f"{type(exc).__name__}: {str(exc)}"
        )[:1500]

    except Exception as exc:

        record["status"] = "unexpected_error"
        record["error"] = (
            f"{type(exc).__name__}: {str(exc)}"
        )[:1500]

    # Always save before moving to the next model
    log_path = (
        TEST_LOG_DIR /
        f"{model_key}_{ROW_ID:03d}_{timestamp_utc()}.json"
    )

    with open(log_path, "x", encoding="utf-8") as f:
        json.dump(
            record,
            f,
            indent=2,
            ensure_ascii=False
        )

    print("\nHTTP:", record["http_status"])
    print("Status:", record["status"])
    print("Raw:", record["raw_response"])
    print("Five words:", record["five_words"])
    print("Valid:", record["valid"])

    if record["error"]:
        print("Error:", record["error"][:600])

    print("Usage:", record["usage"])
    print("Log saved:", log_path)

    return record


# ------------------------------------------------------------
# 10. EXECUTE ALL FOUR MODELS
# ------------------------------------------------------------

print("=" * 65)
print("APPEARANCE AI V2 - FOUR MODEL TEST")
print("=" * 65)
print("Image:", IMAGE_NAME)
print("MD5 verified:", True)
print("Prompt version: V2")
print("Models:", len(EXPECTED_MODELS))
print("=" * 65)

# LIVE API STEP: calls may be sent to Gemini and Cloudflare.
# Existing valid logs are reused unless FORCE_RETEST is enabled.
all_results = []

for model_key in EXPECTED_MODELS:

    model_id = MODELS[model_key]

    result = test_model(
        model_key,
        model_id
    )

    all_results.append(result)

    # Stop additional Cloudflare requests on quota errors.
    if (
        model_key != "gemini"
        and result.get("http_status") in (402, 429)
    ):
        print(
            "\nCloudflare quota/billing limit detected."
            "\nRemaining model tests stopped safely."
        )
        break

    time.sleep(1)


# ------------------------------------------------------------
# 11. CREATE COMPARISON TABLE
# ------------------------------------------------------------

summary_rows = []

for item in all_results:

    summary_rows.append({
        "row_id": item["row_id"],
        "image_name": item["image_name"],
        "model_key": item.get("model_key", (
            "gemini" if item["model"] == MODELS["gemini"]
            else next(
                k for k, v in MODELS.items()
                if v == item["model"]
            )
        )),
        "model": item["model"],
        "http_status": item.get("http_status"),
        "status": item.get("status"),
        "valid": item.get("valid", False),
        "five_words": item.get("five_words", ""),
        "raw_response": item.get("raw_response", ""),
        "error": item.get("error", "")
    })

summary_df = pd.DataFrame(summary_rows)

summary_path = (
    RESULTS_DIR /
    f"four_models_single_001_{timestamp_utc()}.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 65)
print("FINAL FOUR-MODEL COMPARISON")
print("=" * 65)

display(
    summary_df[
        [
            "model_key",
            "http_status",
            "status",
            "valid",
            "five_words",
            "error"
        ]
    ]
)

print("\nTests completed:", len(summary_df))
print("Valid results:", int(summary_df["valid"].sum()))
print("Summary saved:", summary_path)
print("=" * 65)


### 2.5 — Register and execute the external 75-image batch runner
**Type:** **LIVE batch execution** · **Required external file:** `Appearance_AI_V2_Batch75.py`

Upload the Python runner from the original project, save it under `Appearance_AI_V2/scripts`, and execute its contents. This file is **not embedded in the notebook**, so the notebook alone cannot reproduce the full batch. Review the uploaded code before executing it.

In [ ]:

from google.colab import files
from pathlib import Path

# Upload the downloaded Python script
# The original batch runner is an EXTERNAL Python file, not defined here.
# Inspect the uploaded script before this cell executes it.
uploaded = files.upload()

FILENAME = "Appearance_AI_V2_Batch75.py"

if FILENAME not in uploaded:
    raise FileNotFoundError(
        f"Please upload {FILENAME}"
    )

# Store script permanently in Google Drive
SCRIPT_DIR = Path(
    "/content/drive_project_account/MyDrive/"
    "Translucency_AI/Appearance_AI_V2/scripts"
)

SCRIPT_DIR.mkdir(parents=True, exist_ok=True)

SCRIPT_PATH = SCRIPT_DIR / FILENAME

content = uploaded[FILENAME]

if SCRIPT_PATH.exists():
    if SCRIPT_PATH.read_bytes() != content:
        raise FileExistsError(
            "A different script version already exists. "
            "Existing file was not overwritten."
        )
else:
    SCRIPT_PATH.write_bytes(content)

print("Script saved:", SCRIPT_PATH)
print("Starting batch processing...\n")

exec(
    compile(
        SCRIPT_PATH.read_text(encoding="utf-8"),
        str(SCRIPT_PATH),
        "exec"
    )
)


### 2.6 — Optional provider-specific batch variants
**Type:** **LIVE Cloudflare batch** · **Historical recovery path**

Build Cloudflare-only and Gemini-only variants by replacing one verified model-selection line in the original runner. The cell immediately executes the Cloudflare-only variant. Run this only when continuing an incomplete 75-image batch; it is **not** a standard step after a successful full batch.

In [ ]:

# ======================================================
# APPEARANCE AI V2 - CONTINUE WITHOUT GEMINI
# ======================================================

from pathlib import Path
import runpy

SCRIPT_DIR = Path(
    "/content/drive_project_account/MyDrive/"
    "Translucency_AI/Appearance_AI_V2/scripts"
)

ORIGINAL = SCRIPT_DIR / "Appearance_AI_V2_Batch75.py"

if not ORIGINAL.is_file():
    raise FileNotFoundError(
        f"Original batch script missing: {ORIGINAL}"
    )

source = ORIGINAL.read_text(encoding="utf-8")

old_setting = (
    'ACTIVE_MODELS = ["gemini", "qwen", "llama", "mistral"]'
)

if source.count(old_setting) != 1:
    raise RuntimeError(
        "Original model configuration not found exactly once. "
        "No files changed."
    )

# Create provider-specific runner variants from the original script.
# The code checks for an exact source string before applying substitutions.
variants = {
    "CloudflareOnly": (
        'ACTIVE_MODELS = ["qwen", "llama", "mistral"]'
    ),
    "GeminiOnly": (
        'ACTIVE_MODELS = ["gemini"]'
    )
}

saved_scripts = {}

for label, new_setting in variants.items():

    target = SCRIPT_DIR / (
        f"Appearance_AI_V2_Batch75_{label}.py"
    )

    patched = source.replace(
        old_setting,
        new_setting,
        1
    )

    if target.exists():
        if target.read_text(encoding="utf-8") != patched:
            raise FileExistsError(
                f"Different file already exists: {target}"
            )
    else:
        target.write_text(
            patched,
            encoding="utf-8"
        )

    saved_scripts[label] = target
    print("Ready:", target.name)

print("\n" + "=" * 65)
print("RESUMING CLOUDFLARE MODELS")
print("Gemini is excluded from this run.")
print("Previous valid responses will be reused.")
print("=" * 65)

runpy.run_path(
    str(saved_scripts["CloudflareOnly"]),
    run_name="__main__"
)


### 2.7 — Optional Cloudflare quota-profile resume
**Type:** **LIVE Cloudflare batch** · **Historical recovery path**

Construct `Appearance_AI_V2_Batch75_Cloudflare10000.py` from the original 75-image runner with the specific model, quota ceiling, buffer, and retry settings recorded in the cell. This represents an experiment-specific historical setting; it does not automatically select a different Cloudflare account or guarantee the provider's current quota.

In [ ]:

# ======================================================
# APPEARANCE AI V2
# RESUME CLOUDFLARE - 10000 NEURONS
# ======================================================

from pathlib import Path

SCRIPT_DIR = Path(
    "/content/drive_project_account/MyDrive/"
    "Translucency_AI/Appearance_AI_V2/scripts"
)

ORIGINAL = SCRIPT_DIR / "Appearance_AI_V2_Batch75.py"

NEW_SCRIPT = SCRIPT_DIR / (
    "Appearance_AI_V2_Batch75_Cloudflare10000.py"
)

if not ORIGINAL.is_file():
    raise FileNotFoundError(
        f"Original script not found: {ORIGINAL}"
    )

source = ORIGINAL.read_text(encoding="utf-8")

# Modify only the required settings.
# Historical quota-specific batch variant: adjusts model selection, quota,
# retry, and safety-buffer settings. This is not an API credential switch.
changes = {
    'ACTIVE_MODELS = ["gemini", "qwen", "llama", "mistral"]':
        'ACTIVE_MODELS = ["mistral", "qwen"]',

    'CF_DAILY_RUN_CEILING = 7500.0':
        'CF_DAILY_RUN_CEILING = 10000.0',

    'CF_BUFFER_FOR_NEXT_REQUEST = 200.0':
        'CF_BUFFER_FOR_NEXT_REQUEST = 100.0',

    'RETRY_INVALID_FORMAT = False':
        'RETRY_INVALID_FORMAT = True',
}

for old, new in changes.items():
    if source.count(old) != 1:
        raise RuntimeError(
            f"Expected exactly one occurrence: {old}"
        )
    source = source.replace(old, new, 1)

# Validate Python syntax before saving.
compile(source, str(NEW_SCRIPT), "exec")

# Do not overwrite a different script.
if NEW_SCRIPT.exists():
    if NEW_SCRIPT.read_text(encoding="utf-8") != source:
        raise FileExistsError(
            f"Different script already exists: {NEW_SCRIPT}"
        )
else:
    NEW_SCRIPT.write_text(source, encoding="utf-8")

print("=" * 60)
print("CLOUDFLARE BATCH SETTINGS")
print("=" * 60)
print("Models: Mistral, Qwen")
print("Daily ceiling: 10000 Neurons")
print("Safety buffer: 100 Neurons")
print("Retry invalid responses: True")
print("Resume previous successes: True")
print("Script:", NEW_SCRIPT)
print("=" * 60)

# Start batch processing.
%run "/content/drive_project_account/MyDrive/Translucency_AI/Appearance_AI_V2/scripts/Appearance_AI_V2_Batch75_Cloudflare10000.py"


### 2.8 — Optional Gemini-only resume with safety checks
**Type:** **LIVE Gemini batch** · **Historical recovery path**

Verify that `Appearance_AI_V2_Batch75_GeminiOnly.py` enables only Gemini and includes a prior-success protection setting, then execute it. Use only if there are unfinished Gemini responses.

In [ ]:

# ============================================================
# APPEARANCE AI V2 - RESUME GEMINI
# 75 images | Prompt V2 | Gemini only
# ============================================================

from google.colab import drive
from pathlib import Path
import os

MOUNT_POINT = "/content/drive_project_account"

# 1. Connect Drive if necessary
if not os.path.ismount(MOUNT_POINT):
    drive.mount(MOUNT_POINT)

# 2. Find the Gemini-only batch script
SCRIPT_PATH = Path(
    "/content/drive_project_account/MyDrive/"
    "Translucency_AI/Appearance_AI_V2/scripts/"
    "Appearance_AI_V2_Batch75_GeminiOnly.py"
)

if not SCRIPT_PATH.is_file():
    raise FileNotFoundError(
        f"Gemini batch script not found: {SCRIPT_PATH}"
    )

# 3. Verify that only Gemini is enabled
# Validate the Gemini-only script and prior-success protection
# before a live batch resume.
source = SCRIPT_PATH.read_text(encoding="utf-8")

if 'ACTIVE_MODELS = ["gemini"]' not in source:
    raise RuntimeError(
        "This script is not configured for Gemini only."
    )

if "FORCE_RETEST_SUCCESS = False" not in source:
    raise RuntimeError(
        "Successful-result protection is not enabled."
    )

print("=" * 65)
print("RESUMING GEMINI - APPEARANCE AI V2")
print("=" * 65)
print("Dataset       : 75 images")
print("Prompt        : V2")
print("Model         : Gemini")
print("Reuse success : Enabled")
print("Script        :", SCRIPT_PATH.name)
print("=" * 65)

# 4. Execute existing batch without changing its settings
%run /content/drive_project_account/MyDrive/Translucency_AI/Appearance_AI_V2/scripts/Appearance_AI_V2_Batch75_GeminiOnly.py


### 2.9 — Direct Gemini batch rerun (duplicate historical entry point)
**Type:** **LIVE Gemini batch** · **Optional — normally skip**

This shorter `%run` repeats the same external Gemini-only script without the preceding safeguards. It is retained as part of the original execution history, **not** as an additional required step. Prefer the verified resume cell above.

In [ ]:

# HISTORICAL DIRECT RERUN: may send API requests.
# Use only when intentionally resuming the 75-image Gemini batch.
%run /content/drive_project_account/MyDrive/Translucency_AI/Appearance_AI_V2/scripts/Appearance_AI_V2_Batch75_GeminiOnly.py


### 2.10 — Recover an absent Gemini-only script
**Type:** **LIVE Gemini batch** · **Optional recovery**

If the Gemini-specific script exists, verify it. Otherwise construct it from `Appearance_AI_V2_Batch75.py` only when the expected model setting and success-protection option are present. The final action runs the verified script. Do not execute this after the same work is complete.

In [ ]:

# ============================================================
# APPEARANCE AI V2 - RESTORE & RESUME GEMINI
# Keeps all previous successful results
# ============================================================

from pathlib import Path
from google.colab import drive
import os

MOUNT = Path("/content/drive_project_account")

# 1. Reconnect Google Drive if required
if not os.path.ismount(str(MOUNT)):
    drive.mount(str(MOUNT))

SCRIPT_DIR = (
    MOUNT / "MyDrive/Translucency_AI/"
    "Appearance_AI_V2/scripts"
)

ORIGINAL = SCRIPT_DIR / "Appearance_AI_V2_Batch75.py"
GEMINI_ONLY = (
    SCRIPT_DIR / "Appearance_AI_V2_Batch75_GeminiOnly.py"
)

print("=" * 65)
print("GEMINI RESUME - SCRIPT CHECK")
print("=" * 65)

if not SCRIPT_DIR.is_dir():
    raise FileNotFoundError(
        f"Scripts folder not found: {SCRIPT_DIR}"
    )

print("Available Python files:")
for p in sorted(SCRIPT_DIR.glob("*.py")):
    print(" -", p.name)

# 2. Use existing Gemini script or recreate it
# Prefer an existing Gemini-only script; otherwise derive a verified copy
# from the original runner without overwriting unrelated versions.
if GEMINI_ONLY.is_file():
    print("\nGemini-only script already exists.")
    code = GEMINI_ONLY.read_text(encoding="utf-8")

else:
    if not ORIGINAL.is_file():
        raise FileNotFoundError(
            "Original batch script is missing too. "
            "Please check the available filenames above."
        )

    code = ORIGINAL.read_text(encoding="utf-8")

    original_setting = (
        'ACTIVE_MODELS = ["gemini", "qwen", "llama", "mistral"]'
    )

    if code.count(original_setting) != 1:
        raise RuntimeError(
            "The original model settings differ from "
            "the expected version. Nothing was changed."
        )

    code = code.replace(
        original_setting,
        'ACTIVE_MODELS = ["gemini"]',
        1
    )

    # Ensure we never repeat successful results
    if "FORCE_RETEST_SUCCESS = False" not in code:
        raise RuntimeError(
            "Success protection was not found."
        )

    compile(code, str(GEMINI_ONLY), "exec")

    # Never overwrite an existing script
    with GEMINI_ONLY.open("x", encoding="utf-8") as f:
        f.write(code)

    print("\nGemini-only script created.")

# 3. Final safety checks
if 'ACTIVE_MODELS = ["gemini"]' not in code:
    raise RuntimeError("Script is not Gemini-only.")

if "FORCE_RETEST_SUCCESS = False" not in code:
    raise RuntimeError("Success protection is disabled.")

compile(code, str(GEMINI_ONLY), "exec")

print("\n" + "=" * 65)
print("READY TO RESUME")
print("=" * 65)
print("Model: Gemini only")
print("Dataset: images.zip (75 images)")
print("Prompt: V2")
print("Skip previous successes: YES")
print("Script:", GEMINI_ONLY)
print("=" * 65)

# 4. Run the verified Gemini-only script
%run /content/drive_project_account/MyDrive/Translucency_AI/Appearance_AI_V2/scripts/Appearance_AI_V2_Batch75_GeminiOnly.py


### 2.11 — Inspect alternate Cloudflare secret availability
**Type:** Offline credential check · **API inference:** No

Check whether the primary and secondary Cloudflare key/account ID pairs are accessible through Colab Secrets. The code prints **availability booleans only**, not key values. These are optional account-specific historical settings.

In [ ]:

# ================================================
# APPEARANCE AI - SECOND CLOUDFLARE ACCOUNT CHECK
# No API requests
# ================================================

from google.colab import userdata

# Availability check only: read booleans, never print secret values.
# The alternate Cloudflare account requires its own credential pair.
secret_names = [
    "Flare_api_key",
    "Flare_account_id",
    "Flare2_api_key",
    "Flare2_account_id"
]

print("=" * 55)
print("CLOUDFLARE CONNECTION CONFIGURATION")
print("=" * 55)

for name in secret_names:
    try:
        value = userdata.get(name)
        ready = bool(value and str(value).strip())
        print(f"{name:<22}: {ready}")
    except Exception as e:
        print(f"{name:<22}: False ({type(e).__name__})")

print("=" * 55)


### 2.12 — Diagnose remaining 75-image responses
**Type:** Offline audit · **Input:** `appearance_images75_v2_needs_review.csv`

Read previously saved review records, isolate Qwen and Mistral cases, and print their status, HTTP code, original response, and error. This cell helps decide whether resuming is necessary; it does not send requests.

In [ ]:

# ============================================================
# APPEARANCE AI V2 - CHECK REMAINING 12 RESPONSES
# No API requests
# ============================================================

from pathlib import Path
import pandas as pd

REPORT = Path(
    "/content/drive_project_account/MyDrive/"
    "Translucency_AI/Appearance_AI_V2/results/"
    "appearance_images75_v2_needs_review.csv"
)

if not REPORT.is_file():
    raise FileNotFoundError(REPORT)

# Read the saved needs-review report to diagnose incomplete responses.
# This cell does not call a model or change the dataset.
df = pd.read_csv(
    REPORT,
    dtype=str,
    keep_default_na=False
)

remaining = df[
    df["model_key"].isin(["qwen", "mistral"])
].copy()

remaining["row_id"] = remaining["row_id"].astype(int)

remaining = remaining.sort_values(
    ["model_key", "row_id"]
)

print("=" * 70)
print("REMAINING QWEN / MISTRAL RESPONSES")
print("=" * 70)

print("Total:", len(remaining))

print("\nStatus breakdown:")
print(
    remaining.groupby(
        ["model_key", "status"]
    ).size().to_string()
)

print("\n" + "=" * 70)

for _, row in remaining.iterrows():

    print(f"Model  : {row['model_key']}")
    print(f"Image  : {row['image_name']}")
    print(f"Status : {row['status']}")
    print(f"HTTP   : {row['http_status']}")
    print(f"Raw    : {repr(row['raw_response'])}")
    print(f"Error  : {row['error']}")
    print("-" * 70)


### 2.13 — Historical audit-column helper (not standalone)
**Type:** Offline · **Status:** **Optional / requires manual context**

This original helper validates and normalizes an existing pandas DataFrame named **`audit`**. **The cell does not load that variable**, and it will raise `NameError` in a fresh runtime. It is preserved for provenance, not recommended for a top-to-bottom run. Supply the intended audit DataFrame explicitly before using it; do not substitute a different experiment's audit automatically.

In [ ]:
# ============================================================
# STEP 3 - BASIC NORMALIZATION
# ============================================================

# HISTORICAL HELPER: assumes a pandas DataFrame named `audit`
# has already been created in the active session. Not standalone.
required_cols = [
    "row_id",
    "image_name",
    "model_key",
    "status",
    "valid",
    "raw_response",
    "five_words",
    "http_status",
    "error"
]

for col in required_cols:
    if col not in audit.columns:
        raise ValueError(
            f"Missing required column: {col}"
        )

audit["row_id"] = audit["row_id"].astype(int)
audit["valid"] = audit["valid"].astype(str)

optional_cols = [
    "model",
    "usage",
    "api_response",
    "prompt_version",
    "prompt_sha256",
    "image_md5",
    "timestamp",
    "dataset"
]

for col in optional_cols:
    if col not in audit.columns:
        audit[col] = ""

print("AUDIT VALIDATION COMPLETED")
print("Total rows:", len(audit))
print("Total columns:", len(audit.columns))

### 2.14 — Install optional Excel export dependency
**Type:** Package setup · **API inference:** No

Install `XlsxWriter`, which is needed by the offline Excel organizer. This is not a model execution step.

In [ ]:

# Install the missing Excel library

# Excel export support; install this only when generating workbooks.
%pip install -q "XlsxWriter>=3.2,<4"

import xlsxwriter

print("XlsxWriter installed successfully.")
print("Version:", xlsxwriter.__version__)


### 2.15 — Export the Part 1 analysis workbook
**Type:** Offline processing · **Required external file:** `organize_part1_75.py`

Execute the separate organizer under the Part 1 scripts directory, locate the latest generated `Appearance_Part1_75_Organized.xlsx`, and offer it for download. The 75-image workflow represents up to **300 model/image response slots** and is distinct from the 90-image analysis.

In [ ]:

# ============================================================
# APPEARANCE AI V2 - PART 1 FINAL EXCEL EXPORT
# 75 images | 4 models | 300 response slots
# No API requests
# ============================================================

import subprocess
import sys
from pathlib import Path
from google.colab import files

ROOT = Path(
    "/content/drive_project_account/MyDrive/"
    "Translucency_AI/Appearance_AI_V2"
)

# EXTERNAL OFFLINE EXPORT: runs an organizer already saved on Drive.
# No model inference is requested by this notebook cell.
SCRIPT_PATH = ROOT / "scripts/organize_part1_75.py"
RESULTS_DIR = ROOT / "results"

if not SCRIPT_PATH.is_file():
    raise FileNotFoundError(SCRIPT_PATH)

print("=" * 65)
print("GENERATING FINAL EXCEL - PART 1")
print("=" * 65)

result = subprocess.run(
    [
        sys.executable,
        str(SCRIPT_PATH),
        "--results-dir",
        str(RESULTS_DIR)
    ],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Excel export failed.")

excel_files = sorted(
    RESULTS_DIR.glob(
        "organized_part1_75/export_*/"
        "Appearance_Part1_75_Organized.xlsx"
    ),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

if not excel_files:
    raise FileNotFoundError("Excel output not found.")

EXCEL_PATH = excel_files[0]

print("=" * 65)
print("EXCEL EXPORT SUCCESSFUL")
print("=" * 65)
print("File:", EXCEL_PATH)
print("Size:", round(EXCEL_PATH.stat().st_size / 1024, 2), "KB")
print("=" * 65)

files.download(str(EXCEL_PATH))


---
## 3. Part 2 — Main 90-image experiment

This is the 90-image four-model study relevant to the paper's aggregate response statistics. The next two validators address the same setup; they are documented separately because both appeared in the original notebook.

### 3.1 — Primary manifest/config builder for the 90-image dataset
**Type:** Offline validation · **Expected input:** `Translucency_AI/input/images2.zip`

Build independent folders, manifest, and prompt/model configuration for **Part 2 (90 images)**, verifying archive integrity, image numbering, dimensions, and checksums. It writes `manifest_images90.csv` and `experiment_config_images90.json` with non-overwrite checks.

**Important:** the next cell is a second historical implementation of essentially the same validation task; **choose one**, rather than routinely executing both.

In [ ]:

# ============================================================
# APPEARANCE AI V2 - PART 2
# STEP 1: DATASET VALIDATION AND MANIFEST
#
# Dataset: images2.zip
# Expected images: 90
# Prompt: V2
# No API requests
# ============================================================

from pathlib import Path, PurePosixPath
from zipfile import ZipFile
from collections import Counter
from io import BytesIO
from PIL import Image
from google.colab import drive

import pandas as pd
import hashlib
import json
import os
import re

# ------------------------------------------------------------
# 1. GOOGLE DRIVE
# ------------------------------------------------------------

MOUNT_POINT = "/content/drive_project_account"

if not os.path.ismount(MOUNT_POINT):
    drive.mount(MOUNT_POINT)

# ------------------------------------------------------------
# 2. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive_project_account/MyDrive/Translucency_AI"
)

ZIP_PATH = PROJECT_ROOT / "input/images2.zip"

PART2_ROOT = PROJECT_ROOT / "Appearance_AI_V2_Part2_90"

LOGS_DIR = PART2_ROOT / "logs"
RESULTS_DIR = PART2_ROOT / "results"
SCRIPTS_DIR = PART2_ROOT / "scripts"

MANIFEST_PATH = LOGS_DIR / "manifest_images90.csv"
CONFIG_PATH = LOGS_DIR / "experiment_config_images90.json"

# Alternative Part 2 manifest builder. It verifies image IDs, dimensions,
# checksums, prompt/config metadata, and protects existing manifests.
EXPECTED_COUNT = 90

# ------------------------------------------------------------
# 3. PROMPT V2
# ------------------------------------------------------------

PROMPT_V2 = (
    "Describe the visual appearance of the main object or "
    "material in this image using exactly five English "
    "descriptive words.\n\n"
    "Rules:\n"
    "- Output exactly five words\n"
    "- Separate them with commas\n"
    "- No explanation\n"
    "- No full sentence"
)

PROMPT_HASH = hashlib.sha256(
    PROMPT_V2.encode("utf-8")
).hexdigest()

MODELS = {
    "gemini": "gemini-3.5-flash-lite",
    "qwen": "@cf/qwen/qwen3.8-27b",
    "llama": "@cf/meta/llama-4-scout-17b-16e-instruct",
    "mistral": "@cf/mistralai/mistral-small-3.1-24b-instruct"
}

# ------------------------------------------------------------
# 4. VERIFY ZIP
# ------------------------------------------------------------

if not ZIP_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset ZIP not found: {ZIP_PATH}"
    )

VALID_EXTENSIONS = {
    ".jpg", ".jpeg", ".jfif", ".png", ".webp"
}

rows = []

with ZipFile(ZIP_PATH, "r") as zf:

    corrupted = zf.testzip()

    if corrupted is not None:
        raise ValueError(
            f"Corrupted ZIP member: {corrupted}"
        )

    image_members = []

    for info in zf.infolist():

        if info.is_dir():
            continue

        path = PurePosixPath(info.filename)

        if "__MACOSX" in path.parts:
            continue

        if path.suffix.lower() not in VALID_EXTENSIONS:
            continue

        image_members.append(info)

    print("=" * 65)
    print("APPEARANCE AI V2 - PART 2 DATASET CHECK")
    print("=" * 65)

    print("ZIP:", ZIP_PATH.name)
    print("Images detected:", len(image_members))
    print("Expected images:", EXPECTED_COUNT)

    if len(image_members) != EXPECTED_COUNT:
        raise ValueError(
            f"Expected {EXPECTED_COUNT} images, "
            f"found {len(image_members)}."
        )

    image_ids = []

    for info in image_members:

        image_path = PurePosixPath(info.filename)
        stem = image_path.stem

        if not re.fullmatch(r"\d+", stem):
            raise ValueError(
                f"Image filename is not numeric: {info.filename}"
            )

        image_ids.append(int(stem))

    duplicate_ids = [
        image_id
        for image_id, count in Counter(image_ids).items()
        if count > 1
    ]

    expected_ids = set(range(1, EXPECTED_COUNT + 1))
    actual_ids = set(image_ids)

    missing_ids = sorted(expected_ids - actual_ids)
    unexpected_ids = sorted(actual_ids - expected_ids)

    if duplicate_ids or missing_ids or unexpected_ids:

        print("Duplicate IDs:", duplicate_ids)
        print("Missing IDs:", missing_ids)
        print("Unexpected IDs:", unexpected_ids)

        raise ValueError("Image numbering validation failed.")

    # --------------------------------------------------------
    # 5. VERIFY EACH IMAGE
    # --------------------------------------------------------

    for info in image_members:

        image_path = PurePosixPath(info.filename)
        image_id = int(image_path.stem)

        image_bytes = zf.read(info.filename)

        image_md5 = hashlib.md5(
            image_bytes
        ).hexdigest()

        with Image.open(BytesIO(image_bytes)) as image:

            width, height = image.size
            image_format = image.format

            image.verify()

        rows.append({
            "row_id": image_id,
            "image_name": image_path.name,
            "relative_path": info.filename,
            "md5": image_md5,
            "size_bytes": len(image_bytes),
            "width": width,
            "height": height,
            "image_format": image_format
        })

# ------------------------------------------------------------
# 6. BUILD MANIFEST
# ------------------------------------------------------------

manifest_df = pd.DataFrame(rows)

manifest_df = manifest_df.sort_values(
    "row_id"
).reset_index(drop=True)

if manifest_df["row_id"].tolist() != list(range(1, 91)):
    raise ValueError("Image sequence validation failed.")

# ------------------------------------------------------------
# 7. BUILD INDEPENDENT CONFIG
# ------------------------------------------------------------

config = {
    "experiment": "Appearance_AI_V2_Part2_90",
    "dataset": "images90",
    "image_count": EXPECTED_COUNT,
    "zip_file": ZIP_PATH.name,
    "prompt_version": "V2",
    "prompt": PROMPT_V2,
    "prompt_sha256": PROMPT_HASH,
    "models": MODELS
}

# ------------------------------------------------------------
# 8. SAFE SAVE
# ------------------------------------------------------------

LOGS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

if MANIFEST_PATH.exists():

    existing = pd.read_csv(
        MANIFEST_PATH,
        dtype={
            "image_name": str,
            "relative_path": str,
            "md5": str,
            "image_format": str
        }
    )

    if not existing.equals(manifest_df):
        raise RuntimeError(
            "Existing manifest differs from the current ZIP. "
            "No files were overwritten."
        )

    print("\nExisting manifest verified.")

else:

    manifest_df.to_csv(
        MANIFEST_PATH,
        index=False,
        encoding="utf-8-sig"
    )

    print("\nNew manifest saved.")

if CONFIG_PATH.exists():

    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        existing_config = json.load(f)

    if existing_config != config:
        raise RuntimeError(
            "Existing experiment configuration differs. "
            "No configuration was overwritten."
        )

    print("Existing configuration verified.")

else:

    with open(CONFIG_PATH, "x", encoding="utf-8") as f:
        json.dump(
            config,
            f,
            indent=2,
            ensure_ascii=False
        )

    print("New experiment configuration saved.")

# ------------------------------------------------------------
# 9. FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PART 2 - DATASET VALIDATION SUMMARY")
print("=" * 65)

print("Total images :", len(manifest_df))
print("First image  :", manifest_df.iloc[0]["image_name"])
print("Last image   :", manifest_df.iloc[-1]["image_name"])
print("Unique MD5   :", manifest_df["md5"].nunique())

print("\nManifest:", MANIFEST_PATH)
print("Config  :", CONFIG_PATH)

print("\nRegistered models:")

for name, model in MODELS.items():
    print(f"  {name:<10}: {model}")

print("\nExpected model responses:", 90 * len(MODELS))

print("\nSTATUS: PART 2 DATASET READY")
print("=" * 65)

display(manifest_df.head(10))


### 3.2 — Alternative 90-image manifest builder (historical duplicate)
**Type:** Offline validation · **Alternative to §3.1**

This version repeats the Part 2 archive/manifest/config checks with slightly different implementation details. It is kept because it was part of the research record. On a new project, use **one validated implementation** to initialize the 90-image experiment; the other is available for inspection, not a required second processing pass.

In [ ]:

# ============================================================
# APPEARANCE AI - PART 2
# STEP 1: DATASET 90 VALIDATION
# No API requests
# ============================================================

from pathlib import Path, PurePosixPath
from zipfile import ZipFile
from collections import Counter
from io import BytesIO
from PIL import Image
from google.colab import drive

import pandas as pd
import hashlib
import json
import os
import re

# ------------------------------------------------------------
# 1. GOOGLE DRIVE
# ------------------------------------------------------------

MOUNT_POINT = "/content/drive_project_account"

if not os.path.ismount(MOUNT_POINT):
    drive.mount(MOUNT_POINT)

# ------------------------------------------------------------
# 2. INDEPENDENT PART 2 PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive_project_account/MyDrive/Translucency_AI"
)

ZIP_PATH = PROJECT_ROOT / "input/images2.zip"

PART2_ROOT = PROJECT_ROOT / "Appearance_AI_V2_Part2_90"

LOGS_DIR = PART2_ROOT / "logs"
RESULTS_DIR = PART2_ROOT / "results"
SCRIPTS_DIR = PART2_ROOT / "scripts"

MANIFEST_PATH = LOGS_DIR / "manifest_images90.csv"
CONFIG_PATH = LOGS_DIR / "experiment_config_images90.json"

# Second historical implementation of Part 2 dataset validation.
# It targets the same 90-image manifest/config as the preceding alternative.
EXPECTED_COUNT = 90

# ------------------------------------------------------------
# 3. PROMPT V2
# ------------------------------------------------------------

PROMPT_V2 = (
    "Describe the visual appearance of the main object or "
    "material in this image using exactly five English "
    "descriptive words.\n\n"
    "Rules:\n"
    "- Output exactly five words\n"
    "- Separate them with commas\n"
    "- No explanation\n"
    "- No full sentence"
)

PROMPT_HASH = hashlib.sha256(
    PROMPT_V2.encode("utf-8")
).hexdigest()

MODELS = {
    "gemini": "gemini-3.5-flash-lite",
    "qwen": "@cf/qwen/qwen3.8-27b",
    "llama": "@cf/meta/llama-4-scout-17b-16e-instruct",
    "mistral": "@cf/mistralai/mistral-small-3.1-24b-instruct"
}

# ------------------------------------------------------------
# 4. VERIFY ZIP
# ------------------------------------------------------------

if not ZIP_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found: {ZIP_PATH}"
    )

VALID_EXTENSIONS = {
    ".jpg", ".jpeg", ".jfif", ".png", ".webp"
}

rows = []

with ZipFile(ZIP_PATH, "r") as zf:

    damaged = zf.testzip()

    if damaged is not None:
        raise ValueError(
            f"Damaged ZIP member: {damaged}"
        )

    image_members = []

    for info in zf.infolist():

        if info.is_dir():
            continue

        image_path = PurePosixPath(info.filename)

        if "__MACOSX" in image_path.parts:
            continue

        if image_path.suffix.lower() not in VALID_EXTENSIONS:
            continue

        image_members.append(info)

    print("=" * 65)
    print("PART 2 - DATASET VALIDATION")
    print("=" * 65)

    print("ZIP:", ZIP_PATH.name)
    print("Images detected:", len(image_members))
    print("Expected:", EXPECTED_COUNT)

    if len(image_members) != EXPECTED_COUNT:
        raise ValueError(
            f"Expected {EXPECTED_COUNT} images, "
            f"found {len(image_members)}"
        )

    image_ids = []

    for info in image_members:

        stem = PurePosixPath(info.filename).stem

        if not re.fullmatch(r"\d+", stem):
            raise ValueError(
                f"Non-numeric image filename: {info.filename}"
            )

        image_ids.append(int(stem))

    duplicates = [
        image_id
        for image_id, count in Counter(image_ids).items()
        if count > 1
    ]

    missing = sorted(
        set(range(1, 91)) - set(image_ids)
    )

    unexpected = sorted(
        set(image_ids) - set(range(1, 91))
    )

    if duplicates or missing or unexpected:

        print("Duplicates:", duplicates)
        print("Missing:", missing)
        print("Unexpected:", unexpected)

        raise ValueError("Image numbering validation failed.")

    # --------------------------------------------------------
    # 5. VERIFY EACH IMAGE
    # --------------------------------------------------------

    for info in image_members:

        path = PurePosixPath(info.filename)
        image_id = int(path.stem)

        image_bytes = zf.read(info.filename)

        md5 = hashlib.md5(image_bytes).hexdigest()

        with Image.open(BytesIO(image_bytes)) as image:
            width, height = image.size
            image_format = image.format
            image.verify()

        rows.append({
            "row_id": image_id,
            "image_name": path.name,
            "relative_path": info.filename,
            "md5": md5,
            "size_bytes": len(image_bytes),
            "width": width,
            "height": height,
            "image_format": image_format
        })

# ------------------------------------------------------------
# 6. BUILD MANIFEST
# ------------------------------------------------------------

manifest_df = pd.DataFrame(rows)

manifest_df = (
    manifest_df
    .sort_values("row_id")
    .reset_index(drop=True)
)

if manifest_df["row_id"].tolist() != list(range(1, 91)):
    raise ValueError("Image sequence is invalid.")

# ------------------------------------------------------------
# 7. SAVE INDEPENDENT PART 2 FILES
# ------------------------------------------------------------

LOGS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

if MANIFEST_PATH.exists():

    existing = pd.read_csv(MANIFEST_PATH)

    if not existing.equals(manifest_df):
        raise RuntimeError(
            "Existing manifest differs. "
            "No files were overwritten."
        )

    print("\nExisting manifest verified.")

else:

    manifest_df.to_csv(
        MANIFEST_PATH,
        index=False,
        encoding="utf-8-sig"
    )

    print("\nNew manifest saved.")

config = {
    "experiment": "Appearance_AI_V2_Part2_90",
    "dataset": "images90",
    "image_count": 90,
    "zip_file": "images2.zip",
    "prompt_version": "V2",
    "prompt": PROMPT_V2,
    "prompt_sha256": PROMPT_HASH,
    "models": MODELS
}

if CONFIG_PATH.exists():

    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        existing_config = json.load(f)

    if existing_config != config:
        raise RuntimeError(
            "Existing configuration differs."
        )

    print("Existing configuration verified.")

else:

    with open(CONFIG_PATH, "x", encoding="utf-8") as f:
        json.dump(
            config,
            f,
            indent=2,
            ensure_ascii=False
        )

    print("New configuration saved.")

# ------------------------------------------------------------
# 8. FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PART 2 - FINAL DATASET SUMMARY")
print("=" * 65)

print("Total images:", len(manifest_df))
print("First image:", manifest_df.iloc[0]["image_name"])
print("Last image:", manifest_df.iloc[-1]["image_name"])
print("Unique MD5:", manifest_df["md5"].nunique())

print("\nManifest:", MANIFEST_PATH)
print("Config:", CONFIG_PATH)

print("\nModels:")
for name, model in MODELS.items():
    print(f"{name:<10}: {model}")

print("\nExpected responses:", 90 * len(MODELS))

print("\nSTATUS: PART 2 READY")
print("=" * 65)

display(manifest_df.head(10))


### 3.3 — Upload the external 90-image runner
**Type:** Offline script registration · **Required file:** `part2_batch90.py`

Accept one external script, compile it for syntax checking, and save it in the Part 2 Drive scripts directory. The file is **not included in this notebook**, and the cell does not itself perform model inference.

In [ ]:

from google.colab import drive, files
from pathlib import Path
import os

MOUNT = "/content/drive_project_account"

if not os.path.ismount(MOUNT):
    drive.mount(MOUNT)

# Upload the external 90-image batch runner and verify its syntax
# before saving. This cell does not execute it.
uploaded = files.upload()

matches = [
    (name, content)
    for name, content in uploaded.items()
    if name.startswith("part2_batch90")
    and name.endswith(".py")
]

if len(matches) != 1:
    raise ValueError("Upload exactly one part2_batch90.py file.")

name, content = matches[0]

SCRIPT_DIR = Path(MOUNT) / (
    "MyDrive/Translucency_AI/"
    "Appearance_AI_V2_Part2_90/scripts"
)

SCRIPT_DIR.mkdir(parents=True, exist_ok=True)

SCRIPT_PATH = SCRIPT_DIR / "part2_batch90.py"

compile(content.decode("utf-8"), str(SCRIPT_PATH), "exec")

if SCRIPT_PATH.exists():
    if SCRIPT_PATH.read_bytes() != content:
        raise FileExistsError("A different script already exists.")
else:
    SCRIPT_PATH.write_bytes(content)

print("PART 2 SCRIPT READY")
print("Path:", SCRIPT_PATH)


### 3.4 — Smoke test the Part 2 runner
**Type:** External runner · **Mode:** `smoke`

Invoke the 90-image runner's smoke-test mode before batch execution. Since its implementation is external, inspect that script to determine whether this mode actually sends a test API request.

In [ ]:

# Smoke-test entry point of the external Part 2 runner.
# Check that script's --mode smoke behavior before running it.
%run /content/drive_project_account/MyDrive/Translucency_AI/Appearance_AI_V2_Part2_90/scripts/part2_batch90.py --mode smoke


### 3.5 — Start the 90-image batch
**Type:** **LIVE API batch** · **Mode:** `batch`

Invoke the external batch runner. Expected design of the experiment: **90 images × four models = 360 model/image responses** and **five descriptors per valid response = 1,800 descriptors**. These are expected totals, not a guarantee of successful completion; verify the final audit before reporting them.

In [ ]:
# LIVE BATCH ENTRY POINT: may consume API quota and create logs.
# Run only after the manifest/config and the smoke test are checked.
%run /content/drive_project_account/MyDrive/Translucency_AI/Appearance_AI_V2_Part2_90/scripts/part2_batch90.py --mode batch

### 3.6 — Inspect Qwen HTTP 429 failures
**Type:** Offline diagnosis · **API inference:** No

Search raw response logs for Qwen requests that returned HTTP 429 (rate/quota limiting), sort by timestamp, and show the latest diagnostic record. This cell supports informed resuming instead of blind retries.

In [ ]:

import json
from pathlib import Path

LOG_DIR = Path(
    "/content/drive_project_account/MyDrive/"
    "Translucency_AI/Appearance_AI_V2_Part2_90/"
    "logs/model_responses"
)

# Diagnose logged HTTP 429 rate-limit responses for Qwen.
# Reads existing JSON logs only; no API calls are made.
errors = []

for path in LOG_DIR.glob("*.json"):
    with path.open("r", encoding="utf-8") as f:
        record = json.load(f)

    if (
        record.get("model_key") == "qwen"
        and record.get("http_status") == 429
    ):
        errors.append(record)

errors.sort(key=lambda x: x.get("timestamp", ""))

print("=" * 60)
print("CLOUDFLARE QWEN - HTTP 429 DIAGNOSIS")
print("=" * 60)

if errors:
    latest = errors[-1]
    print("Image:", latest.get("image_name"))
    print("Time:", latest.get("timestamp"))
    print("Error:", latest.get("error"))
    print("API errors:", latest.get("api_response", {}).get("errors"))
else:
    print("No HTTP 429 Qwen logs found.")


---
## 4. Provider-specific resume and quota recovery

These are **alternative operational recovery routes**, developed during experiment execution. They are not a linear workflow. If the first batch completes successfully, these cells generally need **not** be run. Consult the audit and choose the one matching the incomplete provider.

### 4.1 — Resume Gemini and Cloudflare independently
**Type:** **LIVE API resume** · **Required external file:** `part2_batch90_split.py`

Restore the saved Part 2 paths and secrets, recover the provider-split Python runner if needed, verify existing artifacts, then execute `gemini` and `cloudflare` providers separately. Existing logs are checked to reduce the risk of losing previously successful results.

**Run only after inspecting the audit for missing or invalid entries.** This is a historical recovery path, not an additional mandatory run after the main batch.

In [ ]:

# ============================================================
# APPEARANCE AI V2 - PART 2
# RESTORE COLAB SESSION AND RESUME PROCESSING
#
# Dataset: 90 images
# Google Gemini API: Independent
# Cloudflare Workers AI: Independent
# Resume: Enabled
# ============================================================

from google.colab import drive, userdata, files
from pathlib import Path
import subprocess
import sys
import os

# ------------------------------------------------------------
# STEP 1 - RESTORE GOOGLE DRIVE
# ------------------------------------------------------------

MOUNT_POINT = "/content/drive_project_account"

if not os.path.ismount(MOUNT_POINT):
    drive.mount(MOUNT_POINT)

PROJECT_ROOT = (
    Path(MOUNT_POINT)
    / "MyDrive"
    / "Translucency_AI"
)

PART2_ROOT = PROJECT_ROOT / "Appearance_AI_V2_Part2_90"

SCRIPT_PATH = (
    PART2_ROOT
    / "scripts"
    / "part2_batch90_split.py"
)

print("=" * 70)
print("STEP 1 - GOOGLE DRIVE")
print("=" * 70)
print("Drive mounted:", os.path.ismount(MOUNT_POINT))
print("Project exists:", PROJECT_ROOT.is_dir())
print("Part 2 exists:", PART2_ROOT.is_dir())

if not PART2_ROOT.is_dir():
    raise FileNotFoundError(
        f"Part 2 folder not found: {PART2_ROOT}"
    )

# ------------------------------------------------------------
# STEP 2 - VERIFY GOOGLE COLAB SECRETS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 2 - API SECRET VALIDATION")
print("=" * 70)

REQUIRED_SECRETS = [
    "GEMINI_API",
    "Flare_api_key",
    "Flare_account_id"
]

missing_secrets = []

for secret_name in REQUIRED_SECRETS:

    try:
        secret_value = userdata.get(secret_name)

        if secret_value and str(secret_value).strip():
            print(f"{secret_name}: ACCESS GRANTED")
        else:
            print(f"{secret_name}: EMPTY")
            missing_secrets.append(secret_name)

    except Exception as exc:
        print(
            f"{secret_name}: ACCESS FAILED "
            f"({type(exc).__name__})"
        )
        missing_secrets.append(secret_name)

    finally:
        secret_value = None

if missing_secrets:
    raise RuntimeError(
        "Enable Notebook access for these Colab Secrets: "
        + ", ".join(missing_secrets)
    )

print("\nALL REQUIRED API SECRETS ARE ACCESSIBLE")

# ------------------------------------------------------------
# STEP 3 - VERIFY PART 2 DATASET
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 3 - DATASET VERIFICATION")
print("=" * 70)

REQUIRED_FILES = [
    PROJECT_ROOT / "input/images2.zip",
    PART2_ROOT / "logs/manifest_images90.csv",
    PART2_ROOT / "logs/experiment_config_images90.json"
]

for file_path in REQUIRED_FILES:

    if not file_path.is_file():
        raise FileNotFoundError(file_path)

    print("FOUND:", file_path.name)

# ------------------------------------------------------------
# STEP 4 - RECOVER THE PYTHON SCRIPT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 4 - SCRIPT RECOVERY")
print("=" * 70)

if SCRIPT_PATH.is_file():

    print("Existing script found.")
    print("Path:", SCRIPT_PATH)

else:

    print("Script not found in Google Drive.")
    print("Upload part2_batch90_split.py")

    uploaded = files.upload()

    expected_name = "part2_batch90_split.py"

    if expected_name not in uploaded:
        raise FileNotFoundError(
            f"Upload required: {expected_name}"
        )

    source_bytes = uploaded[expected_name]

    compile(
        source_bytes.decode("utf-8"),
        str(SCRIPT_PATH),
        "exec"
    )

    SCRIPT_PATH.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    SCRIPT_PATH.write_bytes(source_bytes)

    print("Script restored successfully.")

# ------------------------------------------------------------
# STEP 5 - VERIFY PREVIOUS RESPONSE LOGS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 5 - PREVIOUS RESULTS")
print("=" * 70)

LOG_DIR = PART2_ROOT / "logs/model_responses"

AUDIT_PATH = (
    PART2_ROOT
    / "results"
    / "appearance_images90_v2_audit.csv"
)

previous_logs = (
    list(LOG_DIR.glob("*.json"))
    if LOG_DIR.is_dir()
    else []
)

print("Previous response logs:", len(previous_logs))
print("Existing audit file:", AUDIT_PATH.is_file())

if AUDIT_PATH.is_file() and not previous_logs:
    raise RuntimeError(
        "An existing audit file was found, but raw response "
        "logs are missing. Processing stopped to protect "
        "previous results."
    )

# ------------------------------------------------------------
# STEP 6 - RESUME PROVIDERS INDEPENDENTLY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 6 - RESUMING API PROCESSING")
print("=" * 70)

# LIVE RESUME WORKFLOW: processes Gemini and Cloudflare separately.
# Preserves existing logs but may retry invalid or missing responses.
PROVIDERS = [
    "gemini",
    "cloudflare"
]

exit_codes = {}

for provider in PROVIDERS:

    print("\n" + "=" * 70, flush=True)
    print(f"STARTING: {provider.upper()}", flush=True)
    print("=" * 70, flush=True)

    command = [
        sys.executable,
        str(SCRIPT_PATH),
        "--provider",
        provider,
        "--mode",
        "batch",
        "--retry-invalid",
        "--delay",
        "1.5"
    ]

    result = subprocess.run(
        command,
        check=False
    )

    exit_codes[provider] = result.returncode

    print("\n" + "-" * 70, flush=True)
    print(
        f"{provider.upper()} PROCESS EXIT CODE: "
        f"{result.returncode}",
        flush=True
    )
    print("-" * 70, flush=True)

# ------------------------------------------------------------
# STEP 7 - FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PART 2 - RESUME SESSION FINISHED")
print("=" * 70)

for provider, code in exit_codes.items():

    status = (
        "PROCESS COMPLETED"
        if code == 0
        else "PROCESS FAILED"
    )

    print(f"{provider.upper()}: {status}")

print("\nResults folder:")
print(PART2_ROOT / "results")

print("\nReview the batch summaries for API quotas.")
print("=" * 70)


### 4.2 — Resume with a third Cloudflare account (older runner)
**Type:** **LIVE Cloudflare API** · **Required file:** `part2_batch90_cloudflare3.py`

Register the third Cloudflare account's secret names and resume the experiment through an older account-specific runner. These secret *names* do not reveal credentials. This cell is an optional historical route; prefer one deliberate resume strategy rather than executing every alternative.

In [ ]:

# ============================================================
# PART 2 - RESTORE SESSION AND RESUME WITH CLOUDFLARE API 3
# ============================================================

from google.colab import drive, userdata, files
from IPython import get_ipython
from pathlib import Path
import os

# Third Cloudflare account configuration, used for a historical
# quota-limited resume. These are names of Colab secrets, not keys.
API3_KEY_SECRET = "Flare3_api_key"
API3_ACCOUNT_SECRET = "Flare3_account_id"

MOUNT_POINT = "/content/drive_project_account"

# ------------------------------------------------------------
# 1. RESTORE GOOGLE DRIVE
# ------------------------------------------------------------

if not os.path.ismount(MOUNT_POINT):
    drive.mount(MOUNT_POINT)

PART2_ROOT = Path(MOUNT_POINT) / (
    "MyDrive/Translucency_AI/Appearance_AI_V2_Part2_90"
)

SCRIPT_PATH = (
    PART2_ROOT / "scripts/part2_batch90_cloudflare3.py"
)

if not PART2_ROOT.is_dir():
    raise FileNotFoundError(PART2_ROOT)

# ------------------------------------------------------------
# 2. VERIFY API 3 SECRETS
# ------------------------------------------------------------

for secret_name in [API3_KEY_SECRET, API3_ACCOUNT_SECRET]:

    try:
        secret_value = userdata.get(secret_name)
    except Exception as exc:
        raise RuntimeError(
            f"Secret access failed: {secret_name}. "
            "Enable Notebook access in Colab Secrets."
        ) from exc

    if not secret_value or not str(secret_value).strip():
        raise ValueError(f"Missing or empty Secret: {secret_name}")

    print(f"{secret_name}: ACCESS GRANTED")
    del secret_value

# ------------------------------------------------------------
# 3. RESTORE API 3 SCRIPT
# ------------------------------------------------------------

if not SCRIPT_PATH.is_file():

    print("Upload part2_batch90_cloudflare3.py")

    uploaded = files.upload()

    filename = "part2_batch90_cloudflare3.py"

    if filename not in uploaded:
        raise FileNotFoundError(filename)

    content = uploaded[filename]

    compile(content.decode("utf-8"), str(SCRIPT_PATH), "exec")

    SCRIPT_PATH.parent.mkdir(parents=True, exist_ok=True)
    SCRIPT_PATH.write_bytes(content)

print("SCRIPT READY:", SCRIPT_PATH)

# ------------------------------------------------------------
# 4. VERIFY EXISTING RESULTS
# ------------------------------------------------------------

LOG_DIR = PART2_ROOT / "logs/model_responses"

log_count = (
    len(list(LOG_DIR.glob("*.json")))
    if LOG_DIR.exists()
    else 0
)

print("Existing response logs:", log_count)

if log_count == 0:
    raise RuntimeError(
        "No existing response logs found. "
        "Verify the project folder before continuing."
    )

# ------------------------------------------------------------
# 5. RESUME PROCESSING
# ------------------------------------------------------------

print("=" * 70)
print("RESUMING PART 2 USING CLOUDFLARE API 3")
print("=" * 70)

args = (
    f"{SCRIPT_PATH} "
    "--provider cloudflare3 "
    "--mode batch "
    "--retry-invalid "
    "--delay 1.5 "
    f"--key-secret {API3_KEY_SECRET} "
    f"--account-secret {API3_ACCOUNT_SECRET}"
)

get_ipython().run_line_magic("run", args)


### 4.3 — Resilient Cloudflare account-3 runner
**Type:** **LIVE Cloudflare API** · **Required file:** `part2_batch90_cloudflare3_resilient.py`

Prepare the more resilient Cloudflare runner, verify required files and credentials, and process outstanding Qwen/Llama/Mistral requests with a retry policy. Rerunning can consume quota. Verify results through the audit after completion.

In [ ]:

# ============================================================
# PART 2 - RESUME ALL REMAINING CLOUDFLARE API 3 REQUESTS
# ============================================================

from google.colab import drive, userdata, files
from pathlib import Path
import subprocess
import sys
import os

MOUNT = "/content/drive_project_account"

if not os.path.ismount(MOUNT):
    drive.mount(MOUNT)

ROOT = Path(MOUNT) / (
    "MyDrive/Translucency_AI/Appearance_AI_V2_Part2_90"
)

# Resilient third-account runner: verify the uploaded file and
# be aware this batch mode can send additional provider requests.
SCRIPT_NAME = "part2_batch90_cloudflare3_resilient.py"
SCRIPT_PATH = ROOT / "scripts" / SCRIPT_NAME

# Verify existing dataset and logs.

required_paths = [
    ROOT / "logs/manifest_images90.csv",
    ROOT / "logs/experiment_config_images90.json",
    ROOT / "results/appearance_images90_v2_audit.csv"
]

for path in required_paths:
    if not path.is_file():
        raise FileNotFoundError(path)

# Verify third-account API credentials.

for secret_name in ["Flare3_api_key", "Flare3_account_id"]:
    value = userdata.get(secret_name)
    if not value or not str(value).strip():
        raise RuntimeError(
            f"Missing or inaccessible Secret: {secret_name}"
        )
    print(f"{secret_name}: ACCESS GRANTED")
    del value

# Restore the updated script.

if not SCRIPT_PATH.is_file():
    print(f"Please upload {SCRIPT_NAME}")

    uploaded = files.upload()

    if SCRIPT_NAME not in uploaded:
        raise FileNotFoundError(SCRIPT_NAME)

    content = uploaded[SCRIPT_NAME]

    compile(content.decode("utf-8"), SCRIPT_NAME, "exec")

    SCRIPT_PATH.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    SCRIPT_PATH.write_bytes(content)

print("SCRIPT READY:", SCRIPT_PATH)

# Resume all remaining Cloudflare models.

command = [
    sys.executable,
    str(SCRIPT_PATH),
    "--provider", "cloudflare3",
    "--mode", "batch",
    "--models", "qwen,llama,mistral",
    "--retry-invalid",
    "--max-retries", "3",
    "--retry-wait", "12",
    "--max-retry-wait", "90",
    "--delay", "1.5"
]

print("\n" + "=" * 70)
print("RESUMING PART 2 - CLOUDFLARE API 3")
print("=" * 70)

result = subprocess.run(command, check=False)

print("\nPROCESS EXIT CODE:", result.returncode)
print("RESULTS:", ROOT / "results")


### 4.4 — Gemini-only recovery using the resilient runner
**Type:** **LIVE Gemini API** · **Optional alternative**

Use the same resilient runner with the Gemini provider only. This is useful for missing or invalid Gemini responses when Cloudflare results are already satisfactory.

In [ ]:

# ============================================================
# PART 2 - RESUME GEMINI USING GOOGLE API ONLY
# ============================================================

from pathlib import Path
from google.colab import userdata
import subprocess
import sys

# Gemini-only resume against the resilient external runner.
# This is an alternative execution path, not a required next step.
SCRIPT_PATH = Path(
    "/content/drive_project_account/MyDrive/"
    "Translucency_AI/Appearance_AI_V2_Part2_90/"
    "scripts/part2_batch90_cloudflare3_resilient.py"
)

if not SCRIPT_PATH.is_file():
    raise FileNotFoundError(SCRIPT_PATH)

if not userdata.get("GEMINI_API"):
    raise RuntimeError("GEMINI_API secret is unavailable.")

print("=" * 70)
print("RESUMING GEMINI WITH GOOGLE API")
print("=" * 70)

command = [
    sys.executable,
    str(SCRIPT_PATH),
    "--provider", "gemini",
    "--mode", "batch",
    "--retry-invalid",
    "--max-retries", "3",
    "--retry-wait", "12",
    "--max-retry-wait", "90",
    "--delay", "1.5"
]

result = subprocess.run(command, check=False)

print("GEMINI EXIT CODE:", result.returncode)


### 4.5 — Audit completeness before any further requests
**Type:** Offline audit · **Input:** `appearance_images90_v2_audit.csv`

Count the unique valid image IDs for each model. A complete experimental grid has **90 valid responses per model**, i.e., **360 valid pairs** in total. Use this audit to decide which provider, if any, requires additional processing.

In [ ]:

# ============================================================
# PART 2 - FINAL COMPLETENESS CHECK
# ============================================================

from pathlib import Path
import pandas as pd

# Offline completeness check. Expected maximum is 90 images x 4 models
# = 360 valid model/image response pairs.
AUDIT_PATH = Path(
    "/content/drive_project_account/MyDrive/"
    "Translucency_AI/Appearance_AI_V2_Part2_90/"
    "results/appearance_images90_v2_audit.csv"
)

audit = pd.read_csv(
    AUDIT_PATH,
    dtype=str,
    keep_default_na=False
)

required = {"row_id", "model_key", "valid"}

if not required.issubset(audit.columns):
    raise ValueError(
        f"Missing columns: {required - set(audit.columns)}"
    )

audit["is_valid"] = (
    audit["valid"].str.strip().str.lower()
    .isin(["true", "1"])
)

total_valid = 0

print("=" * 60)
print("PART 2 - FINAL COMPLETENESS REPORT")
print("=" * 60)

for model in ["gemini", "qwen", "llama", "mistral"]:

    subset = audit[
        (audit["model_key"] == model) &
        (audit["is_valid"])
    ]

    valid_count = subset["row_id"].nunique()
    total_valid += valid_count

    print(
        f"{model:<10}: {valid_count}/90 valid "
        f"| {90 - valid_count} remaining"
    )

print("-" * 60)
print(f"TOTAL VALID: {total_valid}/360")
print(f"REMAINING: {360 - total_valid}")
print("=" * 60)

if total_valid == 360:
    print("PART 2 COMPLETE - READY FOR EXCEL EXPORT")
else:
    print("PART 2 INCOMPLETE - RESUME IS STILL REQUIRED")


### 4.6 — Alternate third-account Cloudflare batch route
**Type:** **LIVE Cloudflare API** · **Historical alternative**

Another invocation of the resilient Cloudflare runner with explicit image range and retry settings. It overlaps with §4.3. **Do not run both routes indiscriminately:** inspect the logs and quota state, then choose only the one needed.

In [ ]:

# ============================================================
# PART 2 - REAL API EXECUTION
# CLOUDFLARE API 3 - RESUME ALL REMAINING IMAGES
# ============================================================

from google.colab import drive, userdata, files
from IPython import get_ipython
from pathlib import Path
import os

MOUNT = "/content/drive_project_account"

if not os.path.ismount(MOUNT):
    drive.mount(MOUNT)

ROOT = Path(MOUNT) / (
    "MyDrive/Translucency_AI/Appearance_AI_V2_Part2_90"
)

# Historical Cloudflare API 3 execution route.
# Do not run alongside the other resume variants without reviewing logs.
SCRIPT_NAME = "part2_batch90_cloudflare3_resilient.py"
SCRIPT_PATH = ROOT / "scripts" / SCRIPT_NAME

# Verify API 3 credentials.

for name in ["Flare3_api_key", "Flare3_account_id"]:
    value = userdata.get(name)

    if not value or not str(value).strip():
        raise RuntimeError(f"Secret unavailable: {name}")

    print(f"{name}: READY")
    del value

# Restore the script only if necessary.

if not SCRIPT_PATH.is_file():

    print("Upload:", SCRIPT_NAME)
    uploaded = files.upload()

    if SCRIPT_NAME not in uploaded:
        raise FileNotFoundError(SCRIPT_NAME)

    content = uploaded[SCRIPT_NAME]
    compile(content.decode("utf-8"), SCRIPT_NAME, "exec")

    SCRIPT_PATH.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    SCRIPT_PATH.write_bytes(content)

print("\nSCRIPT:", SCRIPT_PATH)

# Run actual Cloudflare API requests in the Colab kernel.

print("\n" + "=" * 70)
print("STARTING REAL API REQUESTS")
print("Provider: Cloudflare API 3")
print("Models: Qwen, Llama, Mistral")
print("Mode: Batch / Resume")
print("=" * 70, flush=True)

arguments = (
    f"{SCRIPT_PATH} "
    "--provider cloudflare3 "
    "--mode batch "
    "--start 1 "
    "--end 90 "
    "--models qwen,llama,mistral "
    "--retry-invalid "
    "--max-retries 3 "
    "--retry-wait 12 "
    "--max-retry-wait 90 "
    "--delay 2"
)

get_ipython().run_line_magic("run", arguments)


---
## 5. Response-format repairs and provenance

These cells concern formatting of **previously generated outputs**. They do not make new predictions. Keep a clear distinction between original model text, normalized five-word output, manual correction, and format validity. File-changing routines should be reviewed carefully before rerunning.

### 5.1 — Normalize selected Mistral formatting failures
**Type:** **Offline data modification** · **API inference:** No

Inspect invalid-format Mistral logs, apply narrowly defined formatting normalization, write correction records, back up affected CSV files, and regenerate reports while keeping the original response text available.

**Research integrity:** normalization must never be interpreted as improved model prediction. Verify each transformation against the raw saved response before accepting it.

In [ ]:

# ============================================================
# PART 2 - REPAIR MISTRAL INVALID FORMATS
# No API requests
# Preserve original JSON logs
# ============================================================

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import importlib.util
import hashlib
import csv
import re
import shutil
import uuid
import os

MOUNT = "/content/drive_project_account"

if not os.path.ismount(MOUNT):
    drive.mount(MOUNT)

ROOT = Path(MOUNT) / (
    "MyDrive/Translucency_AI/Appearance_AI_V2_Part2_90"
)

# OFFLINE DATA REPAIR: standardizes selected invalid Mistral formats.
# Read the normalization logic and backups before writing corrected records.
SCRIPT_PATH = (
    ROOT / "scripts/part2_batch90_cloudflare3_resilient.py"
)

if not SCRIPT_PATH.is_file():
    raise FileNotFoundError(SCRIPT_PATH)

# ------------------------------------------------------------
# 1. LOAD THE EXISTING RUNNER
# ------------------------------------------------------------

spec = importlib.util.spec_from_file_location(
    "part2_existing_runner",
    str(SCRIPT_PATH)
)

runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)

config, manifest, images, prompt_hash = runner.validate_inputs()

latest, successes, records = runner.read_logs(
    manifest,
    config["models"],
    prompt_hash
)

# ------------------------------------------------------------
# 2. SAFE FORMAT NORMALIZATION
# ------------------------------------------------------------

WORD_PATTERN = re.compile(
    r"[A-Za-z]+(?:-[A-Za-z]+)*"
)

def fix_format_only(raw):

    if not isinstance(raw, str) or not raw.strip():
        return None

    text = raw.strip()

    text = re.sub(
        r"^```(?:text|plaintext)?[ \t]*\r?\n",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"(?:\r?\n)?[ \t]*```[ \t]*$",
        "",
        text
    )

    text = text.strip()

    if (
        len(text) >= 2
        and text[0] in ("'", '"')
        and text[-1] == text[0]
    ):
        text = text[1:-1].strip()

    text = re.sub(r"\.[ \t]*$", "", text)

    words = [part.strip() for part in text.split(",")]

    if len(words) != 5:
        return None

    if not all(WORD_PATTERN.fullmatch(word) for word in words):
        return None

    return ", ".join(word.lower() for word in words)

# ------------------------------------------------------------
# 3. FIND INVALID MISTRAL RESPONSES
# ------------------------------------------------------------

candidates = []

for row in manifest:

    row_id = int(row["row_id"])
    key = ("mistral", row_id)

    if key in successes:
        continue

    record = latest.get(key)

    if not record:
        continue

    if record.get("status") != "invalid_format":
        continue

    if str(record.get("http_status")) != "200":
        continue

    candidates.append((row_id, record))

print("=" * 70)
print("MISTRAL FORMAT REPAIR")
print("=" * 70)
print("Invalid-format candidates:", len(candidates))

# ------------------------------------------------------------
# 4. BACK UP EXISTING REPORTS
# ------------------------------------------------------------

RESULTS = ROOT / "results"

STAMP = datetime.now(timezone.utc).strftime(
    "%Y%m%d_%H%M%S"
)

BACKUP_DIR = RESULTS / f"backup_before_format_fix_{STAMP}"

if candidates:

    BACKUP_DIR.mkdir(parents=True, exist_ok=True)

    for filename in [
        "appearance_images90_v2_master.csv",
        "appearance_images90_v2_audit.csv",
        "appearance_images90_v2_needs_review.csv",
        "appearance_images90_v2_words.csv"
    ]:

        source = RESULTS / filename

        if source.is_file():
            shutil.copy2(
                source,
                BACKUP_DIR / filename
            )

# ------------------------------------------------------------
# 5. SAVE NORMALIZED SUCCESS RECORDS
# ------------------------------------------------------------

fixed_rows = []
manual_review = []

for row_id, original in candidates:

    original_raw = original.get("raw_response", "")

    normalized = fix_format_only(original_raw)

    if normalized is None:

        manual_review.append(row_id)

        print(
            f"[MISTRAL {row_id:03d}] "
            "MANUAL REVIEW REQUIRED"
        )
        print("Raw:", repr(original_raw[:250]))

        continue

    corrected = dict(original)

    corrected.update({
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "status": "success",
        "valid": True,
        "raw_response": normalized,
        "five_words": normalized,
        "original_raw_response": original_raw,
        "normalization_applied": True,
        "normalization_type": "format_only",
        "normalization_source_timestamp": original.get(
            "timestamp", ""
        ),
        "normalization_source_sha256": hashlib.sha256(
            original_raw.encode("utf-8")
        ).hexdigest(),
        "usage": {},
        "error": ""
    })

    log_filename = (
        f"mistral_{row_id:03d}_format_fix_"
        f"{uuid.uuid4().hex[:12]}.json"
    )

    log_path = runner.RUN_LOGS / log_filename

    runner.atomic_json(log_path, corrected)

    fixed_rows.append({
        "row_id": row_id,
        "image_name": original.get("image_name", ""),
        "original_raw_response": original_raw,
        "normalized_five_words": normalized,
        "source_timestamp": original.get("timestamp", ""),
        "correction_log": log_filename
    })

    print(
        f"[MISTRAL {row_id:03d}] "
        f"FIXED: {normalized}"
    )

# ------------------------------------------------------------
# 6. SAVE NORMALIZATION REPORT
# ------------------------------------------------------------

REPORT_PATH = RESULTS / "mistral_format_fixes.csv"

columns = [
    "row_id",
    "image_name",
    "original_raw_response",
    "normalized_five_words",
    "source_timestamp",
    "correction_log"
]

existing_rows = []

if REPORT_PATH.is_file():

    with REPORT_PATH.open(
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as handle:

        existing_rows = list(csv.DictReader(handle))

merged = {
    str(row["row_id"]): row
    for row in existing_rows
}

for row in fixed_rows:
    merged[str(row["row_id"])] = row

if merged:
    runner.atomic_csv(
        REPORT_PATH,
        columns,
        list(merged.values())
    )

# ------------------------------------------------------------
# 7. REBUILD ALL REPORTS WITHOUT API REQUESTS
# ------------------------------------------------------------

latest, successes, records = runner.read_logs(
    manifest,
    config["models"],
    prompt_hash
)

runner.build_reports(
    manifest,
    config["models"],
    prompt_hash,
    latest,
    successes
)

mistral_valid = sum(
    ("mistral", row_id) in successes
    for row_id in range(1, 91)
)

print("\n" + "=" * 70)
print("MISTRAL FORMAT REPAIR SUMMARY")
print("=" * 70)

print("Newly repaired:", len(fixed_rows))
print("Remaining manual review:", len(manual_review))
print(f"Mistral valid: {mistral_valid}/90")
print("Original logs: PRESERVED")
print("API requests: 0")
print("Correction report:", REPORT_PATH)

if candidates:
    print("CSV backup:", BACKUP_DIR)

print("=" * 70)


### 5.2 — Manually documented Mistral corrections for specific images
**Type:** **Offline historical correction — changes files** · **Target IDs:** `003`, `028`, `085`

Apply explicit image-specific corrections captured during the original study, create backups, update the associated runner where required, write corrected records, and rebuild reports.

**Do not treat this as a generic automatic repair routine.** Review the stored `CORRECTIONS` and original logs, and run only when reproducing those documented historical decisions. The original API outputs must remain traceable.

In [ ]:

# ============================================================
# PART 2 - FINAL MISTRAL FORMAT REPAIR
# Target images: 003, 028, 085
# No API requests
# Preserve original AI responses and research provenance
# ============================================================

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import importlib.util
import hashlib
import shutil
import uuid
import os

MOUNT = "/content/drive_project_account"

if not os.path.ismount(MOUNT):
    drive.mount(MOUNT)

ROOT = Path(MOUNT) / (
    "MyDrive/Translucency_AI/Appearance_AI_V2_Part2_90"
)

SCRIPT = ROOT / "scripts/part2_batch90_cloudflare3_resilient.py"
RESULTS = ROOT / "results"

if not SCRIPT.is_file():
    raise FileNotFoundError(SCRIPT)

# ------------------------------------------------------------
# 1. EXACT MANUAL CORRECTIONS
# ------------------------------------------------------------

# MANUAL RESEARCH CORRECTION: image-specific historical decisions.
# This code can modify the saved runner and create corrected log records.
CORRECTIONS = {
    3: {
        "original": (
            "Vibrant, textured, iridescent, "
            "overlapping, blue feathers"
        ),
        "corrected": (
            "vibrant, textured, iridescent, "
            "overlapping, blue-feathered"
        ),
        "method": "manual_descriptor_rewrite"
    },
    28: {
        "original": (
            "Transparent, spherical, scattered, "
            "varying sizes, reflective"
        ),
        "corrected": (
            "transparent, spherical, scattered, "
            "multi-sized, reflective"
        ),
        "method": "manual_descriptor_rewrite"
    },
    85: {
        "original": (
            "Shimmering, textured, light blue, "
            "sandy, rippled"
        ),
        "corrected": (
            "shimmering, textured, light-blue, "
            "sandy, rippled"
        ),
        "method": "hyphenation"
    }
}

# ------------------------------------------------------------
# 2. LOAD EXISTING RUNNER
# ------------------------------------------------------------

def load_runner(path, module_name):
    spec = importlib.util.spec_from_file_location(
        module_name, str(path)
    )
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

runner = load_runner(SCRIPT, "mistral_repair_runner")

config, manifest, images, prompt_hash = (
    runner.validate_inputs()
)

latest, successes, records = runner.read_logs(
    manifest,
    config["models"],
    prompt_hash
)

# ------------------------------------------------------------
# 3. VALIDATE ALL CANDIDATES BEFORE MODIFYING FILES
# ------------------------------------------------------------

planned = []
already_valid = []

for row_id, correction in CORRECTIONS.items():

    key = ("mistral", row_id)

    normalized, valid = runner.normalized_five_words(
        correction["corrected"]
    )

    if not valid:
        raise ValueError(
            f"Invalid correction for image {row_id:03d}"
        )

    if key in successes:
        already_valid.append(row_id)
        continue

    original = latest.get(key)

    if original is None:
        raise RuntimeError(
            f"Missing Mistral record for image {row_id:03d}"
        )

    if original.get("status") != "invalid_format":
        raise RuntimeError(
            f"Unexpected status for image {row_id:03d}: "
            f"{original.get('status')}"
        )

    if str(original.get("http_status")) != "200":
        raise RuntimeError(
            f"Expected HTTP 200 for image {row_id:03d}"
        )

    raw = str(original.get("raw_response", "")).strip()

    if raw != correction["original"]:
        raise RuntimeError(
            f"Original response mismatch for image {row_id:03d}. "
            "No correction was applied."
        )

    planned.append((row_id, original, correction, normalized))

print("=" * 70)
print("MISTRAL FINAL REPAIR - PRECHECK")
print("=" * 70)

print("Corrections planned:", len(planned))
print("Already valid:", len(already_valid))

for row_id, original, correction, normalized in planned:
    print(f"{row_id:03d}: {normalized}")

# ------------------------------------------------------------
# 4. BACK UP REPORTS AND RUNNER
# ------------------------------------------------------------

STAMP = datetime.now(timezone.utc).strftime(
    "%Y%m%d_%H%M%S_%f"
)

BACKUP = RESULTS / f"backup_mistral_final_{STAMP}"

if planned:
    BACKUP.mkdir(parents=True, exist_ok=False)

    shutil.copy2(
        SCRIPT,
        BACKUP / SCRIPT.name
    )

    for filename in [
        "appearance_images90_v2_master.csv",
        "appearance_images90_v2_audit.csv",
        "appearance_images90_v2_needs_review.csv",
        "appearance_images90_v2_words.csv"
    ]:
        source = RESULTS / filename

        if source.is_file():
            shutil.copy2(source, BACKUP / filename)

# ------------------------------------------------------------
# 5. KEEP ORIGINAL RAW RESPONSES IN FUTURE REPORTS
# ------------------------------------------------------------

source_code = SCRIPT.read_text(encoding="utf-8")

old_line = "raw = rec.get('raw_response', '') or ''"

new_line = (
    "raw = rec.get("
    "'original_raw_response', "
    "rec.get('raw_response', '')"
    ") or ''"
)

if planned:

    if old_line in source_code:

        if source_code.count(old_line) != 1:
            raise RuntimeError(
                "Unexpected runner source structure."
            )

        patched_code = source_code.replace(
            old_line,
            new_line
        )

        compile(
            patched_code,
            str(SCRIPT),
            "exec"
        )

        temp_script = SCRIPT.with_name(
            SCRIPT.name + "." + uuid.uuid4().hex + ".tmp"
        )

        temp_script.write_text(
            patched_code,
            encoding="utf-8"
        )

        os.replace(temp_script, SCRIPT)

        print("Research raw-response preservation enabled.")

    elif new_line in source_code:

        print("Raw-response preservation already enabled.")

    else:

        raise RuntimeError(
            "Runner report format was not recognized. "
            "The existing reports were not modified."
        )

# ------------------------------------------------------------
# 6. SAVE CORRECTED DERIVED LOGS
# ------------------------------------------------------------

report_rows = []

for row_id, original, correction, normalized in planned:

    original_raw = original["raw_response"]

    corrected_record = dict(original)

    corrected_record.pop("api_response", None)

    corrected_record.update({
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "status": "success",
        "valid": True,
        "http_status": 200,
        "raw_response": normalized,
        "five_words": normalized,
        "original_raw_response": original_raw,
        "normalization_applied": True,
        "normalization_type": correction["method"],
        "normalization_source_timestamp": original.get(
            "timestamp", ""
        ),
        "normalization_source_sha256": hashlib.sha256(
            original_raw.encode("utf-8")
        ).hexdigest(),
        "usage": {},
        "error": ""
    })

    log_name = (
        f"mistral_{row_id:03d}_manual_fix_"
        f"{uuid.uuid4().hex[:12]}.json"
    )

    runner.atomic_json(
        runner.RUN_LOGS / log_name,
        corrected_record
    )

    report_rows.append({
        "row_id": row_id,
        "image_name": original.get("image_name", ""),
        "original_raw_response": original_raw,
        "normalized_five_words": normalized,
        "correction_method": correction["method"],
        "correction_log": log_name
    })

    print(f"[MISTRAL {row_id:03d}] FIXED: {normalized}")

# ------------------------------------------------------------
# 7. REBUILD REPORTS WITH ORIGINAL RAW RESPONSES
# ------------------------------------------------------------

runner = load_runner(
    SCRIPT,
    "mistral_repair_runner_updated"
)

latest, successes, records = runner.read_logs(
    manifest,
    config["models"],
    prompt_hash
)

runner.build_reports(
    manifest,
    config["models"],
    prompt_hash,
    latest,
    successes
)

# ------------------------------------------------------------
# 8. VERIFY RESULTS
# ------------------------------------------------------------

mistral_successes = sum(
    ("mistral", row_id) in successes
    for row_id in range(1, 91)
)

for row_id in CORRECTIONS:

    key = ("mistral", row_id)

    if key not in successes:
        raise RuntimeError(
            f"Correction was not recognized: {row_id:03d}"
        )

# ------------------------------------------------------------
# 9. SAVE CORRECTION REPORT
# ------------------------------------------------------------

REPORT = RESULTS / "mistral_final_manual_fixes.csv"

if report_rows:

    runner.atomic_csv(
        REPORT,
        [
            "row_id",
            "image_name",
            "original_raw_response",
            "normalized_five_words",
            "correction_method",
            "correction_log"
        ],
        report_rows
    )

# ------------------------------------------------------------
# 10. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MISTRAL FINAL FORMAT REPAIR SUMMARY")
print("=" * 70)

print("New corrections:", len(report_rows))
print("Already valid:", len(already_valid))
print(f"Mistral valid: {mistral_successes}/90")
print("Original API logs: PRESERVED")
print("API requests: 0")
print("Research report:", REPORT)
print("Reports rebuilt:", RESULTS)

if planned:
    print("Backup directory:", BACKUP)

if mistral_successes == 90:
    print("MISTRAL COMPLETE - 90/90")
else:
    print("MISTRAL INCOMPLETE")

print("=" * 70)


### 5.3 — Final Gemini-only batch resume (if incomplete)
**Type:** **LIVE Gemini API** · **Optional recovery**

Resume remaining Gemini responses with the resilient runner and an explicit retry policy. This is a later alternative to §4.4. If the audit already shows 90/90 valid Gemini responses, **skip it**.

In [ ]:

# ============================================================
# PART 2 - GEMINI ONLY
# DIRECT GOOGLE API
# RESUME ALL REMAINING IMAGES
# ============================================================

from google.colab import drive, userdata
from IPython import get_ipython
from pathlib import Path
import os

# ------------------------------------------------------------
# 1. RESTORE GOOGLE DRIVE
# ------------------------------------------------------------

MOUNT = "/content/drive_project_account"

if not os.path.ismount(MOUNT):
    drive.mount(MOUNT)

ROOT = Path(MOUNT) / (
    "MyDrive/Translucency_AI/Appearance_AI_V2_Part2_90"
)

# LIVE GEMINI-ONLY RESUME: intended for missing/invalid responses
# after checking the existing Part 2 audit.
SCRIPT_PATH = (
    ROOT / "scripts/part2_batch90_cloudflare3_resilient.py"
)

if not SCRIPT_PATH.is_file():
    raise FileNotFoundError(
        f"Runner script not found: {SCRIPT_PATH}"
    )

# ------------------------------------------------------------
# 2. VERIFY GOOGLE GEMINI API KEY
# ------------------------------------------------------------

try:

    gemini_key = userdata.get("GEMINI_API")

except Exception as exc:

    raise RuntimeError(
        "Enable Notebook access for GEMINI_API."
    ) from exc

if not gemini_key or not str(gemini_key).strip():
    raise RuntimeError("GEMINI_API is missing or empty.")

del gemini_key

print("GEMINI_API: ACCESS GRANTED")

# ------------------------------------------------------------
# 3. EXECUTE GEMINI API REQUESTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PART 2 - GEMINI RESUME")
print("Provider: Google Gemini API")
print("Images: 001 to 090")
print("Cloudflare: DISABLED")
print("=" * 70, flush=True)

arguments = (
    f"{SCRIPT_PATH} "
    "--provider gemini "
    "--mode batch "
    "--start 1 "
    "--end 90 "
    "--retry-invalid "
    "--max-retries 2 "
    "--retry-wait 15 "
    "--max-retry-wait 90 "
    "--delay 2"
)

get_ipython().run_line_magic(
    "run",
    arguments
)


---
## 6. Final dataset export

Generate the 90-image workbook from saved response logs and validated audit files.

### 6.1 — Generate the final 90-image Excel workbook
**Type:** Offline export · **Required external file:** `export_part2_90_excel.py`

Run the separate export script, find `Appearance_Part2_90_FINAL.xlsx`, and offer the workbook for download. This step works with saved results; it does not submit model requests. Use the audit to verify completeness before treating the exported table as a finished experimental dataset.

In [ ]:

# ============================================================
# APPEARANCE AI V2 - PART 2 FINAL EXCEL
# 90 images | 4 models | 360 responses | 1800 words
# Offline processing - No API calls
# ============================================================

%pip install -q XlsxWriter

from google.colab import drive, files
from pathlib import Path
import subprocess
import sys
import os

MOUNT = "/content/drive_project_account"

if not os.path.ismount(MOUNT):
    drive.mount(MOUNT)

ROOT = Path(MOUNT) / (
    "MyDrive/Translucency_AI/Appearance_AI_V2_Part2_90"
)

# OFFLINE FINAL EXPORT: external script builds the 90-image Excel report.
# It does not call the vision-language models.
SCRIPT_NAME = "export_part2_90_excel.py"
SCRIPT_PATH = ROOT / "scripts" / SCRIPT_NAME

if not ROOT.is_dir():
    raise FileNotFoundError(ROOT)

if not SCRIPT_PATH.is_file():

    print("Upload:", SCRIPT_NAME)
    uploaded = files.upload()

    if SCRIPT_NAME not in uploaded:
        raise FileNotFoundError(SCRIPT_NAME)

    content = uploaded[SCRIPT_NAME]

    compile(
        content.decode("utf-8"),
        str(SCRIPT_PATH),
        "exec"
    )

    SCRIPT_PATH.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    SCRIPT_PATH.write_bytes(content)

print("=" * 70)
print("GENERATING PART 2 FINAL EXCEL")
print("=" * 70)

result = subprocess.run(
    [
        sys.executable,
        str(SCRIPT_PATH),
        "--root",
        str(ROOT)
    ],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Excel generation failed.")

excel_files = sorted(
    (ROOT / "results").glob(
        "organized_part2_90_final_*/"
        "Appearance_Part2_90_FINAL.xlsx"
    ),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

if not excel_files:
    raise FileNotFoundError(
        "Final Excel was not created."
    )

EXCEL_PATH = excel_files[0]

print("\nFINAL EXCEL READY")
print("Path:", EXCEL_PATH)
print(
    "Size:",
    round(EXCEL_PATH.stat().st_size / 1024, 2),
    "KB"
)

files.download(str(EXCEL_PATH))


---
## 7. Reproducibility notes and known limitations

### External Python scripts referenced in this notebook

The following filenames appear in the supplied notebook or are generated as variants of existing scripts. **The original notebook does not embed their implementation**, so the complete remote-inference pipeline cannot be reproduced from this file alone:

| Script | Role |
|---|---|
| `Appearance_AI_V2_Batch75.py` | Original Part 1 batch runner |
| `Appearance_AI_V2_Batch75_GeminiOnly.py` / `..._CloudflareOnly.py` / `..._Cloudflare10000.py` | Generated or saved Part 1 retry variants |
| `organize_part1_75.py` | Offline export for Part 1 |
| `part2_batch90.py` | Original Part 2 smoke/batch runner |
| `part2_batch90_split.py` | Part 2 provider-specific resume runner |
| `part2_batch90_cloudflare3.py` | Earlier third-account Cloudflare runner |
| `part2_batch90_cloudflare3_resilient.py` | Later resilient provider runner used in recovery and corrections |
| `export_part2_90_excel.py` | Offline Part 2 Excel exporter |

### What can and cannot be inferred

- A valid five-word response indicates **format compliance**, not that each word matches the image.
- Attribute frequency and coverage are **descriptive summary metrics**. They cannot establish which model is visually most accurate without image-grounded human or reference annotations.
- The 75-image and 90-image experiments have different manifests and must be reported separately.
- Rate limits, supported model IDs, provider-specific options, and API behavior may change over time. Preserve the versions, configuration files, timestamps, and original logs used for each run.
- MD5 in the manifest is used here for file comparison, **not** as a security or authenticity guarantee.
- The original Part 1 audit helper (§2.13) expects an existing in-memory DataFrame and is not standalone.
- The correction routines (§5) are experimental-history records, not routine postprocessing steps to apply to new data.

### GitHub publication checklist

- Keep **only the documented `.ipynb` and source `.py` files** required for the repository.
- Exclude `.venv/`, raw image archives unless redistribution is permitted, private response logs, exported databases, and API secret files.
- Notebook output cells have been cleared in this documented copy to avoid publishing cached logs, local runtime details, or accidental sensitive values.
- Document the precise external runner versions and sampling settings before claiming end-to-end reproducibility.

**Documentation policy:** explanatory Markdown and code comments were added without changing the original executable statements. The notebook has **not** been rerun against external services as part of this documentation pass.
